# VCC 2026 · 2a — random-effects meta-analysis, leaderboard-scored tuning and top-N calling

A copy of `vcc_2026_2` with the post-submission review's edits. The transfer model is unchanged: for each target × gene, the precision-weighted average of the backgrounds that measured it (one vote per background: T cells = CD4_Stim48hr only), weight 1/(SE² + τ²), τ² per gene by DerSimonian–Laird pooled over all panel targets, no extra shrinking; the knocked-down gene uses the mean of the sources' on-target LFC.
What changes is how the Δ becomes cells and how settings are chosen:
- **Selection** uses the leaderboard formula: each exam arm is scored against cell-eval2's generic baseline (0) and a replicate anchor (1), clamped and averaged into a scaled overall (edits 1–2), on 75 H1 targets.
- **Calling**: per knockdown, the top N genes by |Δ| / SE among genes > 5 CPM, optionally only where the backgrounds agree in sign (edits 3–4), each raised to a size a 400-cell test can detect (edit 5).
- Every exam arm also writes per-knockdown DE diagnostics (edit 6); the grid is in two stages (edit 7).
- The ad hoc tests check edits 1 and 5 and record D5 / D6 and the generic-response decision (edits 8–10).

Everything is written under `vcc_2026_2a` folders; `vcc_2026_2` and its submission are untouched.

## 0. Session
Paths, the `run` / `bg` / `status` helpers, and input checks.

In [ ]:
import os, sys, shutil, subprocess, shlex, json
from pathlib import Path
import numpy as np
import pandas as pd
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code' / 'vcc_2026_2a'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_2a'),
           B='gs://vcc2026', PY=sys.executable, WORK=str(V / 'lib' / 'vcc_2026_2a'), SOURCES=str(V / 'lib' / 'sources'),
           SCRIPTS=str(V / 'scripts'))          # PY = this notebook's Python (your venv)
os.environ.update(ENV)
for d in ('CODE', 'LOGS', 'LIB', 'DATA', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_2a/<name>.log."""
    session = 'vcc2a_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc2a_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else 'STOPPED - read the log below')
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
need = ['data/gene_names.csv', 'data/pert_counts.csv', 'data/context_A.h5ad', 'data/context_B.h5ad', 'data/context_C.h5ad',
        'lib/h1/H1_full_2025.h5ad', 'scripts/knockdown_filters.py', 'scripts/build_panel_sources.py']
missing = [f for f in need if not (V / f).exists()]
print('python:', sys.executable)
print('data files: all present' if not missing else f'MISSING input files: {missing}')
print('tmux:', 'OK' if shutil.which('tmux') else 'MISSING -> in the terminal: sudo apt-get install -y tmux')
for m in ['numpy', 'pandas', 'scipy', 'h5py', 'anndata']:
    try:
        __import__(m)
    except ImportError:
        print(f'package {m} MISSING -> in this cell: %pip install {m}')
print('official evaluator:', shutil.which('cell-eval2') or 'MISSING: pip install -r requirements-eval.txt')
H1_CELLS = V / 'lib/h1/H1_full_2025.h5ad'  # raw H1 cells: the exam's real side
LINES = ['A', 'B', 'C']                    # use D/E/F with the final bundle
RUN_NAME = 'vcc_2026_2a_topn'   # new name for every run: results are never overwritten
ENV['H1_CELLS'] = str(H1_CELLS); os.environ['H1_CELLS'] = str(H1_CELLS)

Optional: download the raw inputs from the bucket.

In [ ]:
DOWNLOAD_INPUTS = False
if DOWNLOAD_INPUTS:
    run(r"""
    mkdir -p "$LIB/orion" "$LIB/tcell" "$LIB/h1"
    gcloud storage cp --no-clobber "$B/data/*" "$DATA/"
    gcloud storage cp --no-clobber "$B/orion/*" "$LIB/orion/"
    gcloud storage cp --no-clobber "$B/tcell/*" "$LIB/tcell/"
    gcloud storage cp --no-clobber "$B/h1/*" "$LIB/h1/"
    """)

## 1. Scripts
Run all three cells after editing; they write to `~/vcc/code/vcc_2026_2a/`.

### vcc2_common.py
Shared helpers: seeds, JSON, the >5 CPM gate, count checks, control moments.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2a/vcc2_common.py
"""Shared helpers for the VCC 2026 meta-analysis pipeline.

Imported by:
  vcc2_meta.py      : read_names, save_json
  vcc2_generate.py  : seed_for, save_json, expression_gate, int64_indptr, validate_counts, alignment, moments
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

CONTROL = "non-targeting"


def read_names(path, col):
    """One column of a CSV as a numpy string array (first column if `col` is absent)."""
    df = pd.read_csv(path)
    return np.asarray(df[col] if col in df.columns else df.iloc[:, 0], dtype=str)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=1, default=_json_default))


def seed_for(seed, *keys):
    """Deterministic seed from (seed, keys...): stable across runs and machines (unlike Python's hash())."""
    h = hashlib.sha256(repr((int(seed),) + tuple(str(k) for k in keys)).encode()).digest()
    return int.from_bytes(h[:8], "little")


def expression_gate(mean_cpm):
    """Strict scorer boundary, using arithmetic mean per-cell CPM in real controls."""
    mean = np.asarray(mean_cpm, dtype=float)
    return np.isfinite(mean) & (mean > 5.0)


def int64_indptr(matrix):
    """CSR with an int64 row pointer.

    An h5ad X that is written once and then grown with append() keeps the dtype of its
    FIRST block's indptr. scipy uses int32 for small blocks, which caps the finished file at
    2**31 stored values (merged H1 has ~3.5e9; a 360k-cell submission is ~2.2e9).
    """
    m = matrix.tocsr()
    m.indptr = m.indptr.astype(np.int64)
    return m


def validate_counts(matrix, label="counts"):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix)
    if not (np.isfinite(values).all() and (values >= 0).all() and (values == np.floor(values)).all()):
        raise ValueError(f"{label}: expected finite, nonnegative integer raw counts")


def alignment(source_names, genes):
    """For each of OUR genes, its column in the source (-1 = the source did not measure it)."""
    pos = {g: i for i, g in enumerate(np.asarray(source_names, dtype=str))}
    idx = np.array([pos.get(g, -1) for g in np.asarray(genes, dtype=str)], dtype=np.int64)
    if (idx >= 0).sum() == 0:
        raise ValueError("no genes in common with the provided gene list")
    return idx


def _matrix(a, layer):
    if layer is None:
        return a.X
    if layer in a.layers:
        return a.layers[layer]
    raise KeyError(f"layer {layer!r} not found")


def moments(a, rows, idx, layer=None, block=512):
    """Per-gene statistics over `rows` (cells), streamed `block` cells at a time. Library size = sum over OUR genes.
    Returns arrays aligned to our gene list (NaN where the source lacks the gene)."""
    X = _matrix(a, layer); G = len(idx); ok = idx >= 0; cols = idx[ok]
    s_cnt = np.zeros(ok.sum()); s_cpm = np.zeros(ok.sum()); s_cpm2 = np.zeros(ok.sum()); s_det = np.zeros(ok.sum())
    libs = []
    for b in range(0, len(rows), block):
        r = np.asarray(rows[b:b + block])
        M = X[r]
        M = (M.tocsr() if sp.issparse(M) else sp.csr_matrix(np.asarray(M)))[:, cols].astype(np.float64)
        validate_counts(M)
        lib = np.asarray(M.sum(1)).ravel(); libs.append(lib)
        C = sp.diags(1e6 / np.maximum(lib, 1)) @ M
        s_cnt += np.asarray(M.sum(0)).ravel(); s_cpm += np.asarray(C.sum(0)).ravel()
        s_cpm2 += np.asarray(C.multiply(C).sum(0)).ravel(); s_det += np.asarray((M > 0).sum(0)).ravel()
    n = len(rows); lib = np.concatenate(libs) if libs else np.array([0.0])

    def full(v):
        out = np.full(G, np.nan, dtype=np.float32); out[ok] = v; return out
    if n == 0:
        raise ValueError("no cells to profile")
    mean = s_cpm / n; var = np.maximum(s_cpm2 / n - mean ** 2, 0)
    pooled = s_cnt / max(s_cnt.sum(), 1) * 1e6
    stats = dict(mean_cpm=np.where(ok, 0.0, np.nan), pooled_cpm=full(pooled), bulk_cpm=full(pooled), log_mean=full(np.log1p(pooled)),
                 detection=full(s_det / n), var_cpm=full(var), se_mean=full(np.sqrt(var / n)),
                 n_cells=np.int64(n), median_library=np.float64(np.median(lib)), measured=ok.copy(),
                 legacy_baseline=np.bool_(False))
    stats["mean_cpm"][ok] = mean  # float64 preserves the strict 5-CPM boundary
    return stats

### vcc2_meta.py
Steps 1–5: one vote per background, per-gene τ² (DerSimonian–Laird), precision-weighted average with its SE, on-target LFC, and the calling inputs `re_z`, `n_bg`, `sign_agree` (edit 3).

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2a/vcc2_meta.py
"""Random-effects meta-analysis of the public panel sources: precision-weighted average (Model 2).

Inputs: lib/sources/<NAME>/effects_panel.h5ad from scripts/build_panel_sources.py (575 targets x 18,533 genes,
layers lfc / se, NaN = not available). One source per background: the T cells enter as CD4_Stim48hr only.

  fit      per-gene tau2: DerSimonian-Laird, pooled over every target with >= 2 backgrounds -> model dir
  predict  one delta table per line, in the folder format vcc2_generate.py reads:
             re.npy, re_se.npy         weight = 1/(SE^2 + tau2); prediction = sum(weight x lfc) / sum(weight);
                                       SE = sqrt(1 / sum(weight))
             re_z.npy                  |re| / re_se: how convincing each prediction is (ranks genes for top-N calling)
             n_bg.npy                  backgrounds that measured the gene (float; NaN where none did)
             sign_agree.npy            share of those backgrounds whose lfc has the sign of the weighted mean
             n_backgrounds.npy         backgrounds behind each value; 0 = no information (delta stored as NaN)
             ontarget_lfc.npy          mean of the sources' ontarget_lfc: the knocked-down gene, applied separately
  exam     the held-out background's kept targets that at least one other background also kept
  explain  one target x gene: each background's lfc, SE, weight and share, then the prediction

The knocked-down gene's own column never enters tau2 or the averages. No shrinkage: the average is the prediction.

Example: python vcc2_meta.py fit --sources-dir ~/vcc/lib/sources --genes gene_names.csv --holdout H1 --out models/exam
         python vcc2_meta.py predict --model models/exam --targets exam_targets.csv --lines H1 --out exam
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from vcc2_common import read_names, save_json

SCHEMA = 'vcc_2026_2_meta_v2'
SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1']   # step 1: Stim48hr is the one T-cell vote
MODELS = ('re',)


def background_of(root, name):
    a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad', backed='r')
    try: return str(a.uns['background']), a.obs.copy()
    finally: a.file.close()


def included(root, names, holdout=None):
    """Sources that vote, in order; refuses two sources from one background."""
    chosen, seen = [], {}
    for name in names:
        background, _ = background_of(root, name)
        if background == holdout: continue
        if background in seen: raise ValueError(f'{name} and {seen[background]} are both {background}: one vote per background')
        seen[background] = name; chosen.append((name, background))
    if not chosen: raise ValueError('no source backgrounds remain')
    return chosen


class Panel:
    """Included sources stacked as sources x targets x genes (lfc, se), plus each source's on-target LFC."""
    def __init__(self, root, sources, genes):
        self.names = [n for n, _ in sources]; self.backgrounds = [b for _, b in sources]
        lfc, se, own = [], [], []
        for name in self.names:
            a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad')
            if not np.array_equal(a.var_names.astype(str), genes): raise ValueError(f'{name}: gene order differs from gene_names.csv')
            targets = a.obs_names.astype(str).to_numpy()
            if lfc and not np.array_equal(targets, self.targets): raise ValueError(f'{name}: target order differs between panels')
            self.targets = targets
            kept = (a.obs['status'] == 'kept').to_numpy()
            lfc.append(np.asarray(a.layers['lfc'], np.float32)); se.append(np.asarray(a.layers['se'], np.float32))
            own.append(np.where(kept, a.obs['ontarget_lfc'].to_numpy(float), np.nan))
        self.lfc, self.se, self.ontarget = np.stack(lfc), np.stack(se), np.stack(own)
        gp = {g: i for i, g in enumerate(genes)}
        self.own_col = np.array([gp.get(t, -1) for t in self.targets])
        self.row = {t: i for i, t in enumerate(self.targets)}


def tau2_dl(lfc, se, own_col):
    """Per-gene between-background variance, DerSimonian-Laird pooled over targets.

    For each target with k >= 2 backgrounds: Q = sum w (y - mu_FE)^2 with w = 1/SE^2 (noise alone predicts k - 1)
    and C = sum w - sum w^2 / sum w.  tau2 = max(0, (sum Q - sum (k - 1)) / sum C), floored at 0.
    Returns tau2 (NaN where no target ever had two backgrounds) and the pooled degrees of freedom.
    """
    _, T, G = lfc.shape
    Q, df, C = np.zeros(G), np.zeros(G), np.zeros(G)
    for t in range(T):
        y, v = lfc[:, t].astype(np.float64), se[:, t].astype(np.float64) ** 2
        ok = np.isfinite(y) & np.isfinite(v) & (v > 0)
        if own_col[t] >= 0: ok[:, own_col[t]] = False
        k = ok.sum(0); use = k >= 2
        if not use.any(): continue
        w = np.where(ok, 1 / np.where(ok, v, 1), 0); y = np.where(ok, y, 0)
        sw = w.sum(0); mu = np.divide((w * y).sum(0), sw, out=np.zeros(G), where=sw > 0)
        Q[use] += (w * (y - mu) ** 2).sum(0)[use]; df[use] += k[use] - 1
        C[use] += (sw - np.divide((w ** 2).sum(0), sw, out=np.zeros(G), where=sw > 0))[use]
    tau2 = np.full(G, np.nan); est = C > 0
    tau2[est] = np.maximum(0, (Q[est] - df[est]) / C[est])
    return tau2, df


def combine(lfc, se, tau2):
    """Sources x rows x genes -> (re, re_se, k, sign_agree) per row x gene. k == 0 gives NaN (no information).
    sign_agree is the share of the k backgrounds whose lfc has the same sign as the weighted mean."""
    y = lfc.astype(np.float64); total = se.astype(np.float64) ** 2 + tau2   # SE^2 + tau2
    ok = np.isfinite(y) & np.isfinite(total)
    k = ok.sum(0); y = np.where(ok, y, 0)
    w = np.where(ok, 1 / np.where(ok, total, 1), 0); sw = w.sum(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        mean = (w * y).sum(0) / sw
        agree = (ok & (np.sign(y) == np.sign(mean))).sum(0) / k
        out = (mean, np.sqrt(1 / sw))
    re, re_se = (np.where(k > 0, x, np.nan).astype(np.float32) for x in out)
    return re, re_se, k.astype(np.uint8), np.where(k > 0, agree, np.nan).astype(np.float32)


def fit(sources_dir, genes_csv, output, holdout=None, names=SOURCES):
    genes = read_names(genes_csv, 'gene_name'); sources = included(sources_dir, names, holdout)
    panel = Panel(sources_dir, sources, genes)
    tau2, df = tau2_dl(panel.lfc, panel.se, panel.own_col)
    est = np.isfinite(tau2); fallback = float(np.median(tau2[est]))
    measured_in = np.isfinite(panel.lfc).any(1).sum(0)                  # backgrounds measuring each gene at all
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    np.save(out / 'genes.npy', genes); np.save(out / 'tau2.npy', tau2.astype(np.float32)); np.save(out / 'tau2_df.npy', df)
    q = np.quantile(tau2[est], [0.1, 0.25, 0.5, 0.75, 0.9])
    model = dict(schema=SCHEMA, sources_dir=str(Path(sources_dir).resolve()), genes_csv=str(Path(genes_csv).resolve()),
                 sources=[n for n, _ in sources], backgrounds=[b for _, b in sources], heldout_background=holdout,
                 tau2_estimator='DerSimonian-Laird, pooled over targets with >= 2 backgrounds, own column excluded, floored at 0',
                 tau2_fallback=fallback, tau2_fallback_rule='median tau2 for genes no target measured in 2 backgrounds',
                 genes_estimated=int(est.sum()), genes_fallback=int((~est & (measured_in > 0)).sum()),
                 genes_no_data=int((measured_in == 0).sum()), genes_tau2_zero=int((tau2[est] == 0).sum()),
                 tau2_quantiles=dict(zip(['10%', '25%', '50%', '75%', '90%'], q.round(5).tolist())))
    save_json(out / 'model.json', model)
    print(f'fit: {len(sources)} backgrounds {model["backgrounds"]} (held out: {holdout})')
    print(f'  tau2 estimated for {model["genes_estimated"]:,} genes ({model["genes_tau2_zero"]:,} at 0), '
          f'median {fallback:.4f}; {model["genes_fallback"]:,} genes measured in one background use the median; '
          f'{model["genes_no_data"]:,} genes have no data anywhere')
    return model


def load_model(model_dir):
    m = Path(model_dir); model = json.loads((m / 'model.json').read_text())
    if model.get('schema') != SCHEMA: raise ValueError('incompatible model; fit with this notebook')
    genes = np.load(m / 'genes.npy').astype(str); tau2 = np.load(m / 'tau2.npy').astype(np.float64)
    used = np.where(np.isfinite(tau2), tau2, model['tau2_fallback'])
    panel = Panel(model['sources_dir'], list(zip(model['sources'], model['backgrounds'])), genes)
    return model, genes, used, panel


def predict(model_dir, targets_csv, lines, output, chunk=64):
    model, genes, tau2, panel = load_model(model_dir)
    targets = read_names(targets_csv, 'target_gene')
    if not len(targets): raise ValueError('empty perturbation panel')
    out = Path(output)
    for line in lines:   # an earlier meta prediction is replaced (it is deterministic); anything else is kept
        d = out / line; earlier = d / 'prediction_report.json'
        if d.exists() and any(d.iterdir()) and not (earlier.exists() and json.loads(earlier.read_text()).get('schema') == SCHEMA):
            raise ValueError(f'choose an empty prediction directory: {d}')
        if d.exists():
            for old in [*d.glob('*.npy'), d / 'prediction_report.json']: old.unlink(missing_ok=True)
    T, G = len(targets), len(genes)
    arrays = {name: np.full((T, G), np.nan, np.float32) for name in ['re', 're_se', 'sign_agree']}
    count = np.zeros((T, G), np.uint8); ontarget = np.full(T, np.nan, np.float32)
    rows = np.array([panel.row.get(t, -1) for t in targets]); missing = targets[rows < 0].tolist()
    gp = {g: i for i, g in enumerate(genes)}
    for start in range(0, T, chunk):
        idx = np.arange(start, min(start + chunk, T)); idx = idx[rows[idx] >= 0]
        if not len(idx): continue
        lfc, se = panel.lfc[:, rows[idx]].copy(), panel.se[:, rows[idx]]
        for j, i in enumerate(idx):
            if targets[i] in gp: lfc[:, j, gp[targets[i]]] = np.nan   # step 5: own gene is not averaged
        arrays['re'][idx], arrays['re_se'][idx], k, arrays['sign_agree'][idx] = combine(lfc, se, tau2)
        count[idx] = k
        own = panel.ontarget[:, rows[idx]]; n = np.isfinite(own).sum(0)
        ontarget[idx] = np.divide(np.nan_to_num(own).sum(0), n, out=np.full(len(idx), np.nan), where=n > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        arrays['re_z'] = (np.abs(arrays['re']) / arrays['re_se']).astype(np.float32)   # NaN where re is NaN
    arrays['n_bg'] = np.where(count > 0, count, np.nan).astype(np.float32)
    voters = [[b for b, kept in zip(panel.backgrounds, np.isfinite(panel.ontarget[:, r])) if kept] if r >= 0 else [] for r in rows]
    info = (count > 0).sum(1)
    report = dict(schema=SCHEMA, models=list(MODELS), heldout_background=model['heldout_background'],
                  source_backgrounds=model['backgrounds'], source_names=model['sources'], model_dir=str(Path(model_dir).resolve()),
                  targets=str(Path(targets_csv).resolve()), weighting='1 / (SE^2 + tau2)', shrinkage='none', ontarget='mean of the backgrounds\' ontarget_lfc, kept rows only',
                  no_information='NaN in re (n_backgrounds 0); the generator predicts no change there',
                  not_in_panel=missing, no_source_targets=[t for t, v in zip(targets.tolist(), voters) if not v],
                  coverage=[dict(target=t, backgrounds=v, genes_with_information=int(n)) for t, v, n in zip(targets.tolist(), voters, info)])
    for line in lines:
        d = out / line; d.mkdir(parents=True, exist_ok=True)
        np.save(d / 'genes.npy', genes); np.save(d / 'targets.npy', targets)
        for name, value in arrays.items(): np.save(d / f'{name}.npy', value)
        np.save(d / 'n_backgrounds.npy', count); np.save(d / 'ontarget_lfc.npy', ontarget)
        save_json(d / 'prediction_report.json', dict(report, line=line))
    by_k = pd.Series([len(v) for v in voters]).value_counts().sort_index()
    print(f'predict: {T} targets x {G:,} genes -> {", ".join(str(out / l) for l in lines)}')
    print('  targets by number of backgrounds: ' + ', '.join(f'{k}: {n}' for k, n in by_k.items()))
    print(f'  no source at all (all genes "no information"): {report["no_source_targets"] or "none"}')
    if missing: print(f'  not in the panel: {missing}')
    return report


def exam(sources_dir, holdout, output, names=SOURCES):
    status = {}
    for name in names:
        background, obs = background_of(sources_dir, name)
        status.setdefault(background, []).append(obs['status'].eq('kept'))
    if holdout not in status: raise ValueError(f'no source for {holdout}')
    held = pd.concat(status.pop(holdout), axis=1).any(axis=1)
    others = pd.concat([pd.concat(v, axis=1).any(axis=1) for v in status.values()], axis=1).sum(axis=1)
    targets = held.index[held & (others >= 1)].tolist()
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'target_gene': targets}).to_csv(output, index=False)
    print(f'exam: {len(targets)} {holdout} targets also kept in >= 1 other background -> {output}')
    return targets


def explain(model_dir, target, gene):
    """The worked example for one target x gene, from the real panel."""
    if target == gene: raise ValueError('the knocked-down gene is not averaged; see ontarget_lfc')
    model, genes, tau2, panel = load_model(model_dir)
    r, g = panel.row[target], int(np.flatnonzero(genes == gene)[0])
    t2 = tau2[g]; y = panel.lfc[:, r, g].astype(float); se = panel.se[:, r, g].astype(float)
    frame = pd.DataFrame(dict(background=panel.backgrounds, lfc=y, se=se)).set_index('background')
    frame['se2_plus_tau2'] = se ** 2 + t2; frame['weight'] = 1 / frame.se2_plus_tau2
    frame['share'] = frame.weight / frame.weight.sum()
    re, re_se, k, agree = (x[0, 0] for x in combine(frame.lfc.to_numpy()[:, None, None],
                                                                 frame.se.to_numpy()[:, None, None], np.array([t2])))
    summary = dict(target=target, gene=gene, tau2=float(t2), tau2_estimated=bool(np.isfinite(np.load(Path(model_dir) / 'tau2.npy')[g])),
                   n_backgrounds=int(k), re=float(re), re_se=float(re_se), re_z=float(abs(re) / re_se), sign_agree=float(agree))
    return frame, summary


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    f = sub.add_parser('fit')
    for k in ['sources-dir', 'genes', 'out']: f.add_argument('--' + k, required=True)
    f.add_argument('--holdout'); f.add_argument('--sources', nargs='+', default=SOURCES)
    pr = sub.add_parser('predict')
    for k in ['model', 'targets', 'out']: pr.add_argument('--' + k, required=True)
    pr.add_argument('--lines', nargs='+', required=True); pr.add_argument('--chunk', type=int, default=64)
    e = sub.add_parser('exam')
    for k in ['sources-dir', 'holdout', 'out']: e.add_argument('--' + k, required=True)
    e.add_argument('--sources', nargs='+', default=SOURCES)
    x = sub.add_parser('explain')
    for k in ['model', 'target', 'gene']: x.add_argument('--' + k, required=True)
    a = p.parse_args()
    if a.cmd == 'fit': fit(a.sources_dir, a.genes, a.out, a.holdout, a.sources)
    elif a.cmd == 'predict': predict(a.model, a.targets, a.lines, a.out, a.chunk)
    elif a.cmd == 'exam': exam(a.sources_dir, a.holdout, a.out, a.sources)
    else:
        frame, summary = explain(a.model, a.target, a.gene)
        print(frame.to_string(float_format='%.4f')); print(json.dumps(summary, indent=2))

### vcc2_generate.py
Step 6: z or top-N calling with the sign rule, α, detection floor, >5 CPM gate, on-target drop, then cells; the exam benchmark scores every arm with the leaderboard formula and per knockdown (edits 1, 2, 4–7).

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2a/vcc2_generate.py
"""Generate counts with projected composition, metadata cell counts/library sizes,
state-dependent overdispersed induction and reproducible common control sampling.

Finishing: a DE call, then one global alpha, an optional detection floor, the > 5 CPM gate in the line's own controls,
and the knocked-down gene applied separately. Two calling rules:
  z     keep |delta| > z* x SE_pred and |delta| > a minimum size (the vcc_2026_2 submission)
  topn  per knockdown keep the N genes with the largest |delta| / SE_pred among genes > 5 CPM (optionally only genes
        whose backgrounds agree in sign); every other gene is 0
Detection floor: each called gene is raised to the smallest log2 change a 400-cell test against the line's controls
would detect at z_req, so that the genes we call actually show up as DE in the generated cells.

`benchmark` scores arms on the exam (H1-held-out meta-analysis) with official cell-eval2 (vcc2026), streaming every
file one target at a time. The real side and the competition bundle (generic baseline + replicate anchor)
are built once, and can be reused by a later stage (--reference). Every arm is scored with the leaderboard formula
(scaled, clamped, averaged: scaled_scores.csv) and selection uses that overall; raw metrics and ranks are kept.
`oracle` is a separate generator ceiling check and never selects submission knobs.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import shlex
import subprocess
import shutil
import copy
from vcc2_common import int64_indptr, seed_for, validate_counts, save_json, expression_gate, moments, alignment

try:
    from anndata.io import read_elem, write_elem, sparse_dataset
except ImportError:  # older anndata
    from anndata.experimental import read_elem, write_elem, sparse_dataset

VCC2026_METRICS = {
    'pds_cosine': 'max', 'expr_mse_unbiased_capped_norm': 'min',
    'de_wilcoxon_lfc_nmae': 'min', 'de_wilcoxon_direction_fidelity_yield_raw': 'max',
    'de_wilcoxon_direction_reach_raw': 'max', 'de_wilcoxon_sig_jaccard': 'max',
}
CONTROL = "non-targeting"
SCHEMA = "vcc_2026_2_meta_v2"
CC_GENES = ("MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8 HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 "
            "CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 "
            "NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA").split()


class Cfg:
    def __init__(self, a):
        self.alpha = getattr(a, "alpha", 1.0); self.max_lfc = 8.0; self.min_lfc = 0.0
        self.kd_clip = (0.01, 0.9); self.default_kd = 0.30; self.n_cells = getattr(a, "n_cells", 400)
        self.composition = "projection"; self.comp_thr = getattr(a, "comp_thr", 0.02)
        self.pi = getattr(a, "pi", 0.0); self.up_w = getattr(a, "up_w", 0.75)
        self.tilt = getattr(a, "tilt", False); self.match_pool = getattr(a, "match_pool", False) or self.tilt
        self.seed = getattr(a, "seed", 2026)
        self.z = getattr(a, "z", 0.0); self.min_effect = getattr(a, "min_effect", 0.0)   # z call; 0 and 0 = off
        self.call = getattr(a, "call", "z"); self.top_n = int(getattr(a, "top_n", 1000))
        self.sign_rule = bool(getattr(a, "sign_rule", False)); self.detect_floor = bool(getattr(a, "detect_floor", False))
        self.z_req = float(getattr(a, "z_req", 4.0))
        self.metadata = getattr(a, "metadata", True)
        self.library_size = None
        if (not 0 <= self.pi < 1 or not 0 <= self.up_w <= 1 or self.n_cells < 1 or self.alpha < 0
                or self.z < 0 or self.min_effect < 0 or self.call not in ("z", "topn") or self.top_n < 1 or self.z_req <= 0):
            raise ValueError("invalid generator settings")

    def as_dict(self):
        return {k: v for k, v in vars(self).items()}


# ---------------------------------------------------------------- inputs
def read_list(path, col):
    return pd.read_csv(path)[col].astype(str).tolist()


def aligned_X(a, genes):
    """cells x our genes, CSR float32 (missing genes = 0, extra genes dropped)."""
    X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
    names = a.var_names.astype(str).tolist()
    if names == list(genes):
        return X.astype(np.float32)
    pos = {g: i for i, g in enumerate(genes)}
    src = np.array([i for i, g in enumerate(names) if g in pos]); dst = np.array([pos[names[i]] for i in src])
    X = X[:, src].tocoo()
    print(f"  re-indexed genes: {len(src):,}/{len(genes):,} of ours present")
    return sp.csr_matrix((X.data.astype(np.float32), (X.row, dst[X.col])), shape=(X.shape[0], len(genes)))


class Line:
    """Control cells of one context plus the statistics the generator needs."""
    def __init__(self, X, guide, genes):
        validate_counts(X, "control cells")
        self.X = X.tocsr().astype(np.float32); self.genes = genes; self.n = X.shape[0]
        if not self.n or self.X.sum() <= 0: raise ValueError("controls must contain cells and positive counts")
        self.lib = np.asarray(self.X.sum(1)).ravel()
        self.cpm_cells = sp.diags(1e6 / np.maximum(self.lib.astype(np.float64), 1)) @ self.X
        self.cpm = np.asarray(self.cpm_cells.mean(0)).ravel()           # mean per-cell CPM (scorer gate uses > 5)
        self.cpm_var = np.maximum(np.asarray(self.cpm_cells.multiply(self.cpm_cells).mean(0)).ravel() - self.cpm ** 2, 0)
        self.n_reference = self.n                                       # control cells the scorer compares against
        self.share = np.asarray(self.X.sum(0)).ravel() / self.X.sum()   # pooled share of molecules
        self.gate = expression_gate(self.cpm)
        count_mean = np.asarray(self.X.mean(0)).ravel()
        count_var = np.maximum(np.asarray(self.X.multiply(self.X).mean(0)).ravel()-count_mean**2, 0)
        self.overdispersion = np.where(count_mean > 0, np.clip((count_var-count_mean)/np.maximum(count_mean**2, 1e-12), 0, 10), 1.0)
        g = np.asarray(guide).astype(str) if guide is not None else np.zeros(self.n, str)
        self.groups = [np.flatnonzero(g == u) for u in np.unique(g)]
        gp = {x: i for i, x in enumerate(genes)}
        self.cc_idx = np.array([gp[x] for x in CC_GENES if x in gp and self.cpm[gp[x]] > 1], int)
        if len(self.cc_idx) >= 5:
            L = self.cpm_cells[:, self.cc_idx].toarray()
            self.cyc = np.log2(L + 1).mean(1)                            # per-cell cycling level (log2 units)
        else:
            self.cyc = None

    @staticmethod
    def from_h5ad(path, genes, control_col="target_gene"):
        a = ad.read_h5ad(path)
        validate_counts(a.X, str(path))
        if control_col and control_col in a.obs:
            a = a[a.obs[control_col].astype(str).values == CONTROL].copy()
        guide = next((a.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in a.obs), None)
        return Line(aligned_X(a, genes), guide, genes)


def load_deltas(path, targets, genes, array="re", fill=0.0):
    """meta prediction dir (genes.npy, targets.npy, <array>.npy) or .npz (delta [+targets, genes]) -> T x G.

    Missing targets/genes and non-finite values become `fill`: 0 (no change) for deltas, inf (unknown) for SEs.
    """
    p = Path(path)
    if p.is_dir():
        D = np.load(p / f"{array}.npy", mmap_mode="r"); t = np.load(p / "targets.npy").astype(str)
        g = np.load(p / "genes.npy").astype(str)
    else:
        z = np.load(p, allow_pickle=False); D = z["delta"]
        t = z["targets"].astype(str) if "targets" in z else np.array(targets)
        g = z["genes"].astype(str) if "genes" in z else np.array(genes)
    out = np.full((len(targets), len(genes)), fill, np.float32)
    tp = {x: i for i, x in enumerate(t)}; gp = {x: i for i, x in enumerate(g)}
    gsrc = np.array([gp.get(x, -1) for x in genes]); ok = gsrc >= 0
    missing = 0
    for i, x in enumerate(targets):
        if x in tp:
            out[i, ok] = np.asarray(D[tp[x]])[gsrc[ok]]
        else:
            missing += 1
    if missing:
        print(f"  WARNING {path}: {missing} targets missing -> no change predicted for them")
    out[~np.isfinite(out)] = fill
    return out


def load_info(path, targets, genes, model):
    """The calling inputs that exist next to a model's table: se, z (|delta| / SE), n_bg and sign_agree."""
    p = Path(path); info = {}
    for key, name, fill in [("se", model + "_se", np.inf), ("z", model + "_z", np.nan), ("n_bg", "n_bg", np.nan),
                            ("sign_agree", "sign_agree", np.nan)]:
        if p.is_dir() and (p / f"{name}.npy").exists(): info[key] = load_deltas(path, targets, genes, name, fill=fill)
    return info


def load_effects(path, targets, genes, model, cfg):
    """Delta table of one model, plus the calling inputs (SE, z, n_bg, sign_agree) that exist for it."""
    D = load_deltas(path, targets, genes, model)
    info = load_info(path, targets, genes, model)
    if cfg.z > 0 and "se" not in info: raise ValueError(f"{model}: the z call needs {model}_se.npy")
    return D, info


def kd_from_sources(targets, genes, orion_paths=(), tcell_path=None, tcell_condition="Stim48hr", pseudo=0.5):
    """Knocked-down gene's own log2 drop per target, averaged across sources (M1 rule). NaN if no source has it."""
    gp = {g: i for i, g in enumerate(genes)}; vals = [[] for _ in targets]
    for path in orion_paths:
        with h5py.File(path, "r") as f:
            obs = read_elem(f["obs"]); names = list(obs.index.astype(str)); n = obs["n_cells"].values.astype(float)
            var = read_elem(f["var"]); vg = list(var.index.astype(str))
            vpos = {g: i for i, g in enumerate(vg)}
            c = names.index(CONTROL)
            want = [(i, t) for i, t in enumerate(targets) if t in names and t in vpos]
            if not want:
                continue
            rows = sorted({c} | {names.index(t) for _, t in want})
            S = f["layers/sum_cpm"][rows, :]; rpos = {r: k for k, r in enumerate(rows)}
            ctrl = S[rpos[c]] / n[c]
            for i, t in want:
                r = names.index(t); col = vpos[t]
                vals[i].append(np.log2((S[rpos[r], col] / n[r] + pseudo) / (ctrl[col] + pseudo)))
    if tcell_path:
        t = ad.read_h5ad(tcell_path)
        if "condition" in t.obs:
            t = t[t.obs["condition"].astype(str) == tcell_condition]
        tg = t.obs["target_gene"].astype(str).values; tv = {g: i for i, g in enumerate(t.var_names.astype(str))}
        X = np.asarray(t.X)
        for i, x in enumerate(targets):
            r = np.flatnonzero(tg == x)
            if len(r) and x in tv and np.isfinite(X[r[0], tv[x]]):
                vals[i].append(float(X[r[0], tv[x]]))
    return np.array([np.mean(v) if v else np.nan for v in vals])


def kd_residual(log2_drop, cfg):
    r = np.where(np.isfinite(log2_drop), 2.0 ** np.nan_to_num(log2_drop), cfg.default_kd)
    return np.clip(r, *cfg.kd_clip)


# ---------------------------------------------------------------- core generator
def call_top_n(D, line, own, cfg, info):
    """Per knockdown, keep the top_n genes by |delta| / SE among genes > 5 CPM (never the knocked-down gene).
    With sign_rule, a gene measured in >= 2 backgrounds is a candidate only if they all agree in sign."""
    z = info.get("z")
    if z is None:
        if "se" not in info: raise ValueError("the top-N call needs re_z or the prediction SE")
        with np.errstate(divide="ignore", invalid="ignore"): z = np.abs(D) / info["se"]
    cand = line.gate[None, :] & np.isfinite(D) & (D != 0) & np.isfinite(z)
    rows = np.flatnonzero(own >= 0); cand[rows, own[rows]] = False
    if cfg.sign_rule:
        if "n_bg" not in info or "sign_agree" not in info: raise ValueError("the sign rule needs n_bg and sign_agree")
        n_bg, agree = np.asarray(info["n_bg"]), np.asarray(info["sign_agree"])
        cand &= np.isfinite(n_bg) & (n_bg >= 1) & ((n_bg < 2) | (agree == 1))
    score = np.where(cand, z, -np.inf)
    keep = np.zeros_like(cand)
    n = min(cfg.top_n, D.shape[1])
    top = np.argsort(-score, axis=1, kind='stable')[:, :n]   # gene order breaks equal-z ties reproducibly
    np.put_along_axis(keep, top, True, axis=1)
    return np.where(keep & cand, D, 0).astype(np.float32)


def detection_floor(line, cfg, iters=40):
    """Per gene, the smallest |log2 change| (up, down) at which a cfg.n_cells-cell group would differ from the line's
    controls at z_req: m |f - 1| / sqrt(v f^2 / n_cells + v / n_reference) >= z_req, with f = 2^lfc and m, v the
    per-cell CPM mean and variance of the controls (the knockdown keeps the controls' CV). Unreachable -> max_lfc."""
    key = (cfg.n_cells, line.n_reference, cfg.z_req, cfg.max_lfc)
    cache = getattr(line, "_floor", {})
    if key in cache: return cache[key]
    m, v = line.cpm.astype(float), line.cpm_var.astype(float)
    def z_of(l):
        f = np.exp2(l)
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(v > 0, m * np.abs(f - 1) / np.sqrt(v * f ** 2 / cfg.n_cells + v / line.n_reference), np.inf)
    floors = []
    for end in (cfg.max_lfc, -cfg.max_lfc):                      # up, then down
        lo, hi = np.zeros_like(m), np.full_like(m, end)
        reachable = z_of(hi) >= cfg.z_req
        for _ in range(iters):                                    # bisection on the log2 scale between 0 and the cap
            mid = (lo + hi) / 2; ok = z_of(mid) >= cfg.z_req
            hi = np.where(ok, mid, hi); lo = np.where(ok, lo, mid)
        floors.append(np.where(reachable, np.abs(hi), cfg.max_lfc).astype(np.float32))
    cache[key] = tuple(floors); line._floor = cache
    return cache[key]


def calibrate(D, line, targets, cfg, se=None, info=None):
    """DE call, then the single global alpha, the optional detection floor and a binary responder-gene gate.

    z call: an effect stays at full size if |delta| > min_effect and, when z* > 0, |delta| > z* x SE_pred; else 0.
    topn call: see call_top_n. Unknown SE (inf/NaN) is never convincing. No shrinkage of retained effects.
    Detection floor: each called gene becomes sign(delta) x max(alpha |delta|, floor_g).
    The +/-8 generator limit is a numerical safety cap, not an estimated prior.
    On-target suppression is handled separately by measured guide efficiency.
    """
    info = dict(info or {})
    if se is not None: info.setdefault("se", se)
    D = np.asarray(D, np.float32)
    D = np.where(np.isfinite(D), D, 0)
    gp = {g: i for i, g in enumerate(line.genes)}
    own = np.array([gp.get(t, -1) for t in targets])
    if cfg.call == "topn":
        D = call_top_n(D, line, own, cfg, info)
    elif cfg.z > 0 or cfg.min_effect > 0:
        size = np.abs(D); keep = size > cfg.min_effect
        if cfg.z > 0:
            if "se" not in info: raise ValueError("the SE gate (z* > 0) needs the prediction SE")
            with np.errstate(invalid="ignore"):
                keep &= size > cfg.z * np.asarray(info["se"], np.float32)
        D = np.where(keep, D, 0)
    direction = np.sign(D)
    D = (cfg.alpha * D).astype(np.float32)
    if cfg.detect_floor:
        up, down = detection_floor(line, cfg)
        floor = np.where(direction > 0, up[None, :], down[None, :])
        D = np.where(direction != 0, direction * np.maximum(np.abs(D), floor), 0)
    D = np.clip(D, -cfg.max_lfc, cfg.max_lfc).astype(np.float32)
    D[:, ~line.gate] = 0.0
    rows = np.flatnonzero(own >= 0); D[rows, own[rows]] = 0.0
    return D


def pick_cells(line, d, cfg, rng):
    """Step 3 (+ optional tilt): which control cells become this knockdown's 400 cells."""
    n = cfg.n_cells
    if cfg.tilt and line.cyc is not None:
        dcc = float(np.mean(d[line.cc_idx]))                       # predicted cycling shift (log2)
        v = line.cyc - line.cyc.mean(); s = max(v.std(), 1e-6)
        if abs(dcc) > 0.02:
            lo, hi = -30.0, 30.0
            for _ in range(50):                                     # bisection: weighted mean shift == dcc
                b = (lo + hi) / 2; w = np.exp(np.clip(b * v / s, -50, 50)); m = (w * v).sum() / w.sum()
                lo, hi = (b, hi) if m < dcc else (lo, b)
            w = np.exp(np.clip(b * v / s, -50, 50)); w /= w.sum()
            k = min(n, int((w > 0).sum()))
            return rng.choice(line.n, k, replace=False, p=w) if k == n else rng.choice(line.n, n, replace=True, p=w)
    if n > line.n:
        order = np.concatenate([rng.permutation(g) for g in line.groups])
        return np.concatenate([order, rng.choice(line.n, n-line.n, replace=True)])
    per = int(np.ceil(n / len(line.groups)))
    pick = np.concatenate([rng.choice(v, size=min(per, len(v)), replace=False) for v in line.groups])
    if len(pick) >= n:
        return rng.choice(pick, n, replace=False)
    rest = np.setdiff1d(np.arange(line.n), pick)
    return np.concatenate([pick, rng.choice(rest, n - len(pick), replace=len(rest) < n - len(pick))])


def project_factors(share, delta, j=None, kd=None):
    """Project all-gene factors onto the composition simplex without an LFC floor.

    The realized expectation is log2(projected_factor), not the unconstrained
    delta. Zero-baseline induction uses the same 0.5-CPM pseudocount as labels.
    """
    fac = np.exp2(np.asarray(delta, float))
    if j is not None and kd is not None: fac[j] = kd
    proposed = np.asarray(share, float)*fac
    zero_induced = (share == 0) & (fac > 1)
    proposed[zero_induced] = 0.5e-6*(fac[zero_induced]-1)
    mass = float(proposed.sum())
    if not np.isfinite(mass) or mass <= 0: raise ValueError("invalid projected composition")
    projected = proposed/mass
    factors = np.divide(projected, share, out=np.ones_like(projected), where=share > 0)
    return factors, projected, mass


def generate_one(line, d, j, kd, cfg, rng=None, pick=None, line_key="line", target_key="target"):
    """Integer counts, with stable per-gene streams shared between perturbations."""
    if rng is None: rng = np.random.default_rng(seed_for(cfg.seed, line_key, target_key))
    if pick is None:
        # Same base pool per line; target order does not advance a line-level stream.
        pick = pick_cells(line, d, cfg, np.random.default_rng(seed_for(cfg.seed, line_key, "base_controls")))
    B = line.X[pick].toarray().astype(np.int64); d = np.asarray(d, float).copy()
    if cfg.match_pool:
        sampled = np.asarray(line.cpm_cells[pick].mean(0)).ravel()
        deviation = np.log2((sampled+0.5)/(line.cpm+0.5))
        mask = line.gate.copy()
        if j is not None: mask[j] = False
        d[mask] = np.clip(d[mask]-deviation[mask], -cfg.max_lfc, cfg.max_lfc)
    factors, projected, mass = project_factors(line.share, d, j, kd)
    # Library size is an independent whole-cell scale after composition projection.
    lib_scale = 1.0
    if cfg.library_size is not None:
        lib_scale = float(cfg.library_size)/max(float(np.median(line.lib[pick])), 1)
    positive = line.share > 0
    effective_pi = min(cfg.pi, float(factors[positive].min())) if positive.any() else 0.0
    esc_rng = np.random.default_rng(seed_for(cfg.seed, line_key, "escapers"))
    escapers = esc_rng.random(len(pick)) < effective_pi
    cols = np.flatnonzero((np.abs(factors*lib_scale-1) > 1e-12) | ((line.share == 0) & (projected > 0)))
    for g in cols:
        # Fresh per-gene seed gives common random draws without target-order dependence.
        grng = np.random.default_rng(seed_for(cfg.seed, line_key, "gene", line.genes[g]))
        factor = factors[g]
        inside = max((factor-effective_pi)/(1-effective_pi), 0)
        F = np.where(escapers, 1.0, inside)*lib_scale
        original = B[:, g].copy()
        values = grng.binomial(original, np.minimum(F, 1.0))
        up = np.maximum(F-1, 0)
        base = line.lib[pick]*line.share[g]
        if line.share[g] == 0 and projected[g] > 0:
            # Induced genes absent from controls must be able to gain counts.
            intensity = line.lib[pick]*projected[g]*lib_scale
        else:
            expected = cfg.up_w*original+(1-cfg.up_w)*base
            intensity = up*expected
        if np.any(intensity > 0):
            dispersion = float(line.overdispersion[g])
            if dispersion > 1e-8:
                intensity = grng.gamma(1/dispersion, intensity*dispersion)
            values += grng.poisson(intensity)
        B[:, g] = values
    if B.max(initial=0) > np.iinfo(np.int32).max: raise ValueError("count exceeds int32 range")
    out = sp.csr_matrix(B.astype(np.int32)); out.eliminate_zeros(); validate_counts(out)
    planned = np.log2((projected*1e6+0.5)/(line.share*1e6+0.5))
    return out, dict(net_share_change=mass-1, projection_normalizer=mass, n_changed=int(len(cols)),
                     effective_pi=effective_pi, library_scale=lib_scale), planned


def target_metadata(path, line, fallback_n, enabled=True):
    """Optional CSV metadata; supports shared or per-context rows/column suffixes."""
    df = pd.read_csv(path)
    if 'target_gene' not in df: raise ValueError("pert_counts.csv needs target_gene")
    if 'context' in df: df = df[df.context.astype(str) == str(line)]
    if df.target_gene.duplicated().any(): raise ValueError("duplicate target rows; supply context labels")
    result = {}
    for row in df.to_dict('records'):
        n, umi = fallback_n, None
        if enabled:
            for field, default in [('n_cells', fallback_n), ('median_umi_per_cell', None)]:
                value = row.get(field+'_'+str(line), row.get(field, default))
                if value is None or pd.isna(value): value = default
                if value is not None and (not np.isfinite(value) or value <= 0): raise ValueError(f"invalid {field} for {row['target_gene']}")
                if field == 'n_cells':
                    if float(value) != int(value): raise ValueError("n_cells must be integer")
                    n = int(value)
                else: umi = None if value is None else float(value)
        result[str(row['target_gene'])] = dict(n_cells=n, median_umi_per_cell=umi)
    if not result: raise ValueError(f"no requested targets for context {line}")
    return result


def check_output(path, expected_shape, block=512):
    a = ad.read_h5ad(path, backed='r')
    try:
        if a.shape != expected_shape: raise ValueError(f"output shape {a.shape} != {expected_shape}")
        stored = 0
        for start in range(0, a.n_obs, block):
            matrix = a.X[start:start+block]; validate_counts(matrix, f"output rows {start}:{start+block}")
            if np.any(np.asarray(matrix.sum(1)).ravel() > 1e6): raise ValueError('cell exceeds 1,000,000 count cap')
            stored += matrix.nnz if sp.issparse(matrix) else matrix.size
        if a.n_obs > 400000 or stored > 4750000000: raise ValueError('submission exceeds cell or stored-entry cap')
    finally: a.file.close()
    return True


def answer_free_check(cells, line, planned, j):
    pooled = np.asarray(cells.sum(0)).ravel(); pooled = pooled/max(pooled.sum(), 1)*1e6
    realized = np.log2((pooled+0.5)/(line.share*1e6+0.5))
    mask = np.isfinite(realized) & (line.share > 0)
    if j is not None: mask[j] = False
    return dict(projected_effect_mae=float(np.abs(realized[mask]-planned[mask]).mean()),
                n_cells=cells.shape[0], median_umi_per_cell=float(np.median(np.asarray(cells.sum(1)).ravel())))


def prediction_ontarget(path, targets):
    p = Path(path)
    if not p.is_dir() or not (p/'ontarget_lfc.npy').exists(): return np.full(len(targets), np.nan)
    names = np.load(p/'targets.npy').astype(str); values = np.load(p/'ontarget_lfc.npy'); pos = dict(zip(names, values))
    return np.asarray([pos.get(t, np.nan) for t in targets])


def cmd_submit(a):
    cfg = Cfg(a)
    if a.settings:
        settings = json.loads(Path(a.settings).read_text())
        if settings.get('schema') != SCHEMA: raise ValueError('use generator settings from this notebook')
        chosen = settings['settings']
        for key in ['alpha', 'z', 'min_effect', 'call', 'top_n', 'sign_rule', 'detect_floor', 'z_req',
                    'pi', 'up_w', 'n_cells', 'metadata', 'match_pool', 'tilt']:
            if key in chosen: setattr(cfg, key, chosen[key])
        if a.delta_array == 'selected': a.delta_array = settings['delta_array']
    if a.delta_array == 'selected': raise ValueError('--delta-array selected needs --settings')
    data = Path(a.data); genes = read_list(data/'gene_names.csv', 'gene_name')
    gp = {g: i for i, g in enumerate(genes)}; deltas = dict(x.split('=', 1) for x in a.delta)
    specifications = {line: target_metadata(data/'pert_counts.csv', line, cfg.n_cells, cfg.metadata) for line in a.lines}
    sets = [set(s) for s in specifications.values()]
    if any(s != sets[0] for s in sets): raise ValueError('all submitted contexts must share the exact perturbation panel')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True); path = out/'prediction.h5ad'
    if path.exists(): raise FileExistsError('choose a new output directory to preserve the previous submission')
    for line_specs in specifications.values():
        if any(s['n_cells'] != a.required_n_cells for s in line_specs.values()):
            raise ValueError(f'2026 submissions require exactly {a.required_n_cells} cells per target; optional CSV metadata must respect that contract')
    frames = [pd.DataFrame(dict(target_gene=t, context=line), index=[f'{line}_{t}_{i:05d}' for i in range(spec['n_cells'])])
              for line in a.lines for t, spec in specifications[line].items()]
    obs = pd.concat(frames); obs['target_gene'] = obs.target_gene.astype('category'); obs['context'] = obs.context.astype('category')
    report = dict(settings=cfg.as_dict(), delta_array=a.delta_array, lines={})
    with h5py.File(path, 'w') as f:
        write_elem(f, 'obs', obs); write_elem(f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))
        f.attrs['encoding-type'] = 'anndata'; f.attrs['encoding-version'] = '0.1.0'; ds = None
        for name in a.lines:
            if name not in deltas: raise ValueError(f'no predictions for {name}')
            line = Line.from_h5ad(data/f'context_{name}.h5ad', genes)
            targets = list(specifications[name]); D, info = load_effects(deltas[name], targets, genes, a.delta_array, cfg)
            D = calibrate(D, line, targets, cfg, info=info); del info
            own = prediction_ontarget(deltas[name], targets)
            if a.kd_orion or a.kd_tcell:
                legacy_own = kd_from_sources(targets, genes, a.kd_orion or (), a.kd_tcell, a.kd_tcell_condition)
                own = np.where(np.isfinite(own), own, legacy_own)
            kd = kd_residual(own, cfg); checks = []; diags = []
            # Pick max count once; different n_cells use prefixes of this same pool.
            pool_cfg = copy.copy(cfg); pool_cfg.n_cells = max(s['n_cells'] for s in specifications[name].values())
            common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(cfg.seed, name, 'base_controls')))
            for i, t in enumerate(targets):
                local = copy.copy(cfg); local.n_cells = specifications[name][t]['n_cells']; local.library_size = specifications[name][t]['median_umi_per_cell']
                pick = None if local.tilt else common[:local.n_cells]
                cells, diag, planned = generate_one(line, D[i], gp.get(t), kd[i], local, pick=pick, line_key=name, target_key=t)
                diags.append(dict(target=t, **diag))
                if i < a.checks: checks.append(dict(target=t, **answer_free_check(cells, line, planned, gp.get(t))))
                # float32 holds these integer counts exactly and matches vcc prep's output encoding,
                # so prep packages without a second full-matrix copy (int32 needs ~37 GB at 360k cells).
                if cells.data.max(initial=0) >= 2**24: raise ValueError('count too large for exact float32')
                cells = cells.astype(np.float32)
                if ds is None: write_elem(f, 'X', int64_indptr(cells)); ds = sparse_dataset(f['X'])
                else: ds.append(cells)
            pd.DataFrame(checks).to_csv(out/f'checks_{name}.csv', index=False)
            pd.DataFrame(diags).to_csv(out/f'projection_{name}.csv', index=False)
            report['lines'][name] = dict(n_targets=len(targets), n_cells=sum(s['n_cells'] for s in specifications[name].values()),
                metadata=specifications[name], measured_ontarget=int(np.isfinite(own).sum()))
            print(f"generated context {name}", flush=True)
    report['integer_counts'] = check_output(path, (len(obs), len(genes)))
    save_json(out/'generator_report.json', report); print(f'WROTE {path}; checked every output row')


class StreamingH5AD:
    """An .h5ad whose obs is known up front and whose X arrives one block at a time.

    Nothing but the current block is held in memory. The row pointer is int64 from the first
    block, so the finished file may exceed 2**31 stored values. Every block is cast to the first
    block's dtype, and the row total is checked against obs on close.
    """
    def __init__(self, path, obs, genes):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        self.n = len(obs); self.rows = 0; self.ds = None; self.dtype = None
        self.f = h5py.File(self.path, 'w')
        self.f.attrs['encoding-type'] = 'anndata'; self.f.attrs['encoding-version'] = '0.1.0'
        write_elem(self.f, 'obs', obs); write_elem(self.f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))

    def append(self, block):
        block = block.tocsr() if sp.issparse(block) else sp.csr_matrix(np.asarray(block))
        if self.ds is None:
            self.dtype = block.dtype
            write_elem(self.f, 'X', int64_indptr(block), dataset_kwargs={'compression': 'lzf'})
            self.ds = sparse_dataset(self.f['X'])
        else:
            self.ds.append(block.astype(self.dtype, copy=False))
        self.rows += block.shape[0]

    def close(self):
        self.f.close()
        if self.rows != self.n: raise ValueError(f'{self.path}: wrote {self.rows} rows for {self.n} obs entries')


def labelled_obs(labels, prefix):
    labels = np.asarray(labels, dtype=str)
    return pd.DataFrame({'target_gene': pd.Categorical(labels)}, index=[f'{prefix}{i:07d}' for i in range(len(labels))])


def h1_controls(h, labels, genes, seed):
    """The benchmark's seeded control split: the generator half (a Line) and the scorer's half (rows). The gate and
    the detection floor's reference size come from the scorer's half, which is what the evaluator compares against."""
    ci = np.flatnonzero(labels == CONTROL)
    if len(ci) < 2: raise ValueError('H1 needs controls')
    ci = np.random.default_rng(seed_for(seed, 'H1', 'controls')).permutation(ci)
    train_rows, score_rows = ci[:len(ci) // 2], ci[len(ci) // 2:]
    small = h[np.sort(train_rows)].to_memory(); validate_counts(small.X, 'H1 raw counts')
    line = Line(aligned_X(small, genes), None, genes); del small
    # The scorer uses its REAL control half. Match that exact gate, even when
    # the generator half happens to put a near-threshold gene on the other side.
    line.gate = expression_gate(moments(h, np.sort(score_rows), alignment(h.var_names.astype(str), genes))['mean_cpm'])
    line.n_reference = len(score_rows)
    return line, train_rows, score_rows


def reference_targets(real_path):
    """Targets and their real cell counts in an existing scoring reference (a previous stage's real.h5ad)."""
    a = ad.read_h5ad(real_path, backed='r')
    try: counts = a.obs['target_gene'].astype(str).value_counts()
    finally: a.file.close()
    counts = counts.drop(CONTROL)
    return sorted(counts.index), {t: int(n) for t, n in counts.items()}


def stream_h1(a, genes, real_path, line_pseudo=0.5, block=4096):
    """Write the scoring reference straight from the backed H1 file, one target at a time.

    Controls are split into two disjoint, seeded halves: the first half becomes the generator's
    input (held in memory, it is sampled for every target); the second half is appended to the
    reference after the perturbed cells. In oracle mode each target's cells are split the same
    way as before: the estimation half only produces that target's effect vector, the other half
    is appended to the reference. Peak memory is the control half plus one target.

    Returns (Line, targets, {target: reference cell count}, oracle effects or None, oracle own LFCs).
    """
    h = ad.read_h5ad(a.cells, backed='r')
    try:
        labels = h.obs[a.obs_col].astype(str).values
        requested = sorted(set(read_list(a.targets, 'target_gene'))) if a.targets else sorted(set(labels) - {CONTROL})
        counts = pd.Series(labels).value_counts()
        targets = [t for t in requested if counts.get(t, 0) >= a.min_cells and t != CONTROL]
        if a.n_targets and len(targets) > a.n_targets:
            targets = sorted(np.random.default_rng(seed_for(a.seed, 'H1', 'panel')).choice(targets, a.n_targets, replace=False).tolist())
        if not targets: raise ValueError('no eligible H1 targets')

        def subset(rows):
            rows = np.sort(np.asarray(rows))
            small = h[rows].to_memory(); validate_counts(small.X, 'H1 raw counts')
            return aligned_X(small, genes)

        line, train_rows, score_rows = h1_controls(h, labels, genes, a.seed)
        save_json(Path(real_path).parent / 'control_split.json', dict(seed=a.seed,
                  generator_controls=len(train_rows), scoring_controls=len(score_rows),
                  gate_source='scoring real controls', genes_above_5_cpm=int(line.gate.sum())))
        plan = {}
        for t in targets:
            rows = np.flatnonzero(labels == t)
            if a.mode == 'oracle':
                rows = np.random.default_rng(seed_for(a.seed, 'H1', t, 'split')).permutation(rows)
                plan[t] = (rows[:len(rows) // 2], rows[len(rows) // 2:])
            else:
                plan[t] = (None, rows)
        real_counts = {t: len(ref) for t, (_, ref) in plan.items()}
        obs_labels = [t for t in targets for _ in range(real_counts[t])] + [CONTROL] * len(score_rows)
        writer = StreamingH5AD(real_path, labelled_obs(obs_labels, 'real_'), genes)
        oracle, own = ([], []) if a.mode == 'oracle' else (None, None)
        gp = {g: i for i, g in enumerate(genes)}
        try:
            for i, t in enumerate(targets):
                estimation, reference = plan[t]
                if estimation is not None:
                    x = subset(estimation); lib = np.asarray(x.sum(1)).ravel()
                    mean = np.asarray((sp.diags(1e6 / np.maximum(lib, 1)) @ x).mean(0)).ravel()
                    d = np.log2((mean + line_pseudo) / (line.cpm + line_pseudo)); oracle.append(d)
                    own.append(d[gp[t]] if t in gp else np.nan); del x
                writer.append(subset(reference))
                print(f'  reference: {i + 1}/{len(targets)} targets', end='\r', flush=True)
            sorted_score = np.sort(score_rows)
            for start in range(0, len(sorted_score), block):
                writer.append(subset(sorted_score[start:start + block]))
        finally:
            writer.close()
        print(f'\n  wrote {real_path}: {writer.rows:,} cells ({len(score_rows):,} scoring controls)', flush=True)
        return (line, targets, real_counts, None if oracle is None else np.asarray(oracle),
                None if own is None else np.asarray(own))
    finally:
        h.file.close()


def write_arm(path, line, effects, kd, targets, n_cells, common, cfg, line_key='H1', block=4096):
    """Generate each target's cells and append them immediately, then the generator's control cells.

    cell-eval2 requires the predicted file to carry the control label even though vcc2026 scores
    every perturbation against the REAL controls (`control_source: real`), so the generator's own
    input controls are appended as before; they are not used as the scoring reference.
    """
    gp = {g: i for i, g in enumerate(line.genes)}
    labels = [t for t in targets for _ in range(n_cells[t])] + [CONTROL] * line.n
    writer = StreamingH5AD(path, labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            local = copy.copy(cfg); local.n_cells = n_cells[t]
            cells = generate_one(line, effects[i], gp.get(t), kd[i], local, pick=common[:local.n_cells],
                                 line_key=line_key, target_key=t)[0]
            writer.append(cells)
        for start in range(0, line.n, block):
            writer.append(line.X[start:start + block].astype(np.int32))
    finally:
        writer.close()


class OfficialEvaluator:
    """Runs official cell-eval2 against one reference file, and scores arms with the leaderboard formula.

    The reference never changes between arms, so its expensive side (real DE tests, pseudobulks, noise corrections)
    is cached once with `prep-cache --side real` and reused by every arm through `--cache-real`. The ceiling and the
    competition bundle (cell-eval2's generic-response baseline as the 0 end, a real-vs-real replicate anchor as the
    1 end) depend on the real data only, so they are built once too, and a later stage can point at them
    (`reference=` a previous benchmark folder) instead of rebuilding them.
    """
    def __init__(self, a, reference, out, reuse=None):
        self.binary = shutil.which(a.cell_eval)
        if self.binary is None: raise RuntimeError('install official cell-eval2 (pip install -r requirements-eval.txt) before running the H1 benchmark')
        help_text = subprocess.run([self.binary, 'run', '--help'], check=True, text=True, capture_output=True).stdout
        if '--ceiling' not in help_text or '--cache-real' not in help_text or '--write-degenes' not in help_text:
            raise RuntimeError('installed cell-eval2 lacks --ceiling/--cache-real/--write-degenes; install requirements-eval.txt')
        self.a, self.reference, self.out = a, Path(reference), Path(out)
        home = Path(reuse) if reuse else self.out
        self.cache, self.bundle = home / 'real_cache', home / 'bundle'
        self.common = ['--pert-col', 'target_gene', '--control', CONTROL, '--profile', a.eval_profile,
                       '--input-type', 'counts', '--preset', 'vcc2026', '--set', 'de.backend=pdex',
                       '--set', f'num_threads={a.num_threads}']
        self.invocations = []

    def _run(self, command):
        print('  $ ' + ' '.join(shlex.quote(str(x)) for x in command), flush=True)
        subprocess.run([str(x) for x in command], check=True); self.invocations.append([str(x) for x in command])

    def prepare(self):
        # One-sided cache warm: the expression comparator must be named. vcc2026 on counts resolves
        # to bulk_lognorm (cell_eval2.competition.competition_comparator()).
        self._run([self.binary, 'prep-cache', '--side', 'real', '--adata', self.reference,
                   '--cache-real', self.cache, '--comparator', 'bulk_lognorm', *self.common])
        if self.a.mode == 'transfer':
            self._run([self.binary, 'run', '-ar', self.reference, '-o', self.out / 'anchor', '--anchor',
                       '--anchor-splits', str(self.a.anchor_splits), '--cache-real', self.cache, *self.common])
        else:
            self._run([self.binary, 'run', '-ar', self.reference, '-o', self.out / 'ceiling', '--ceiling',
                       '--cache-real', self.cache, *self.common])

    def build_bundle(self):
        """Bind baseline (0) and the cached run --anchor result (1) to this exam's identity.

        In cell-eval2 0.18, score --baseline-agg/--anchor is diagnostic only.
        --real-bundle is required for from_replicate to become the official overall.
        prep-real-bundle reuses the anchor already built by prepare().
        """
        base = self.out / 'baseline'
        self._run([self.binary, 'baseline', '-ar', self.reference, '-o', base, '--save-pred', base / 'baseline_pred.h5ad',
                   '--cache-real', self.cache, *self.common])
        self._run([self.binary, 'prep-real-bundle', '--real', self.reference, '--baseline', base / 'baseline_pred.h5ad',
                   '-o', self.bundle, '--anchor-splits', str(self.a.anchor_splits), '--cache-real', self.cache, *self.common])
        (base / 'baseline_pred.h5ad').unlink()                  # large; the bundle keeps only aggregates
        manifest = json.loads((self.bundle / 'manifest.json').read_text())
        if not manifest.get('rule_digest') or manifest.get('rule_mismatches'):
            raise ValueError(f'not an official competition bundle: {manifest.get("rule_mismatches")}; use five anchor splits')

    def score(self, prediction, output):
        self._run([self.binary, 'run', '-ap', prediction, '-ar', self.reference, '-o', output,
                   '--cache-real', self.cache, '--write-degenes', *self.common])
        self.record(Path(output) / 'invocation.json')

    def scale(self, output):
        """The leaderboard number for one scored arm: cell-eval2 score against the bundle (from_replicate)."""
        output = Path(output)
        self._run([self.binary, 'score', '--user-agg', output / 'agg_results.csv', '--real-bundle', self.bundle,
                   '--user-meta', output / 'run_meta.json', '-o', output / 'scaled.csv'])
        return read_scaled(output / 'scaled.csv')

    def record(self, path):
        import importlib.metadata
        versions = {}
        for package in ['cell-eval2', 'pdex', 'numpy', 'scipy', 'anndata']:
            try: versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: versions[package] = 'not installed in generator environment'
        save_json(path, dict(commands=self.invocations, versions=versions))


def read_scaled(path):
    """A `cell-eval2 score` CSV -> {metric: leaderboard value}, with 'overall' = avg_score (from_replicate)."""
    frame = pd.read_csv(path).set_index('metric')
    expected = set(VCC2026_METRICS) | {'avg_score'}
    if not frame.index.is_unique or set(frame.index) != expected or 'from_replicate' not in frame:
        raise ValueError(f'{path}: expected exactly the six competition metrics and avg_score')
    values = frame['from_replicate'].astype(float).to_dict()
    values['overall'] = values.pop('avg_score')
    if not all(np.isfinite(v) for v in values.values()): raise ValueError(f'{path}: non-finite scaled score')
    if not np.isclose(values['overall'], np.mean([values[k] for k in VCC2026_METRICS]), rtol=1e-8, atol=1e-10):
        raise ValueError(f'{path}: overall is not the mean of the six scaled metrics')
    return values


def per_target_table(arm_dir, alpha=0.05):
    """Per knockdown: our called and the real DE genes (scorer's p_adj < alpha, own gene excluded), their overlap,
    precision against chance (random genes hit real DE genes at n_real / scored genes), direction accuracy, recall."""
    pred = pd.read_parquet(Path(arm_dir) / 'de_pred.parquet'); real = pd.read_parquet(Path(arm_dir) / 'de_real.parquet')
    rows = []
    for t, r in real.groupby('target'):
        p = pred[pred.target == t]
        P = set(p.feature[(p.p_adj < alpha) & (p.feature != t)]); R = set(r.feature[(r.p_adj < alpha) & (r.feature != t)])
        both = P & R; same = 0
        if both:
            lp = p.set_index('feature').log2_fold_change; lr = r.set_index('feature').log2_fold_change
            same = int(sum(np.sign(lp[x]) == np.sign(lr[x]) for x in both))
        scored = r.loc[r.feature != t, 'feature'].nunique(); precision = len(both) / len(P) if P else np.nan
        rows.append(dict(target=t, n_called=len(P), n_real=len(R), overlap=len(both), same_direction=same,
                         precision=precision, chance_precision=len(R) / scored if scored else np.nan,
                         enrichment=precision * scored / len(R) if R else np.nan,
                         direction_accuracy=same / len(both) if both else np.nan, recall=len(both) / len(R) if R else np.nan))
    return pd.DataFrame(rows).set_index('target')


def per_target_summary(table):
    return dict(median_n_called=float(table.n_called.median()), median_n_real=float(table.n_real.median()),
                median_precision=float(table.precision.median()), median_chance_precision=float(table.chance_precision.median()),
                median_enrichment=float(table.enrichment.median()), direction_accuracy=float(table.same_direction.sum() / max(table.overlap.sum(), 1)),
                median_direction_accuracy=float(table.direction_accuracy.median()),
                median_recall=float(table.recall.median()))


def metric_rank_table(outputs, directions_path):
    """Rank only explicitly requested metric families, with explicit directions.

    The evaluator emits diagnostics as well as scoring metrics. Never average
    their ranks blindly. Supply metric,direction ('min'/'max') from current rules.
    """
    directions = pd.read_csv(directions_path)
    if not {'metric', 'direction'} <= set(directions) or directions.metric.duplicated().any():
        raise ValueError('directions CSV needs unique metric,direction columns')
    values = []
    for label, path in outputs.items():
        df = pd.read_csv(Path(path)/'agg_results.csv')
        if not {'metric', 'value'} <= set(df):
            if {'metric', 'score'} <= set(df): df = df.rename(columns={'score': 'value'})
            elif set(directions.metric) <= set(df.columns):
                # cell-eval2 writes a wide describe-style frame; use only its mean row.
                mean_row = np.zeros(len(df), bool)
                for col in df.columns:
                    if col not in set(directions.metric): mean_row |= df[col].astype(str).eq('mean').to_numpy()
                if not mean_row.any() and len(df) == 1: mean_row[:] = True
                if mean_row.sum() != 1: raise ValueError('official aggregate CSV needs exactly one mean row')
                df = pd.DataFrame({'metric': directions.metric, 'value': [df.loc[mean_row, m].iloc[0] for m in directions.metric]})
            else: raise ValueError(f'unrecognized official aggregate schema: {df.columns.tolist()}')
        for metric, direction in directions[['metric', 'direction']].itertuples(index=False, name=None):
            v = df.loc[df.metric == metric, 'value']
            if len(v) != 1 or not np.isfinite(v.iloc[0]): raise ValueError(f'{label}: metric {metric} missing/nonfinite')
            if direction not in ['min', 'max']: raise ValueError('metric direction must be min/max')
            values.append(dict(variant=label, metric=metric, value=float(v.iloc[0]), direction=direction))
    frame = pd.DataFrame(values); frame['rank'] = np.nan
    for metric in directions.metric:
        use = frame.metric == metric
        frame.loc[use, 'rank'] = frame.loc[use, 'value'].rank(ascending=frame.loc[use, 'direction'].iloc[0] == 'min', method='average')
    return frame, frame.groupby('variant')['rank'].mean().sort_values(kind='stable')


ARM_KEYS = ['method', 'call', 'top_n', 'sign_rule', 'detect_floor', 'z_req', 'alpha', 'z', 'min_effect', 'pi', 'up_w', 'cell_rule']


def benchmark_variants(a):
    """The arms to score: an explicit JSON list (--variants) or the product of the grid flags."""
    if a.variants:
        variants = json.loads(Path(a.variants).read_text())
        if not isinstance(variants, list) or not variants: raise ValueError('variants must be a nonempty JSON list')
        labels = set()
        for i, v in enumerate(variants):
            unknown = set(v) - set(ARM_KEYS) - {'label'}
            if unknown or 'method' not in v: raise ValueError(f'bad variant {v}: needs method, unknown keys {sorted(unknown)}')
            v.setdefault('cell_rule', 'fixed')
            label = v.get('label') or f'{v["method"]}_{i:03d}'
            reserved = {'.', '..', 'baseline', 'anchor', 'bundle', 'real_cache', 'ceiling',
                        'benchmark', 'selected_generator', 'reference_info', 'control_split', 'progress'}
            if Path(label).name != label or label in reserved or label in labels:
                raise ValueError(f'arm labels must be unique folder names: {label!r}')
            labels.add(label)
        return variants
    return [dict(method=m, call=a.call, top_n=a.top_n, sign_rule=a.sign_rule, detect_floor=a.detect_floor, z_req=a.z_req,
                 alpha=alpha, z=z, min_effect=me, pi=pi, up_w=w, cell_rule=rule)
            for m in a.delta_array for z in a.z_grid for me in a.min_effect_grid for alpha in a.alpha_grid
            for pi in a.pi_grid for w in a.up_w_grid for rule in a.cell_rules]


def cmd_benchmark(a):
    """Official H1 benchmark, streamed: reference and each arm go to disk one target at a time.

    Every arm is scored raw (agg_results.csv), with the leaderboard formula (scaled.csv) and per knockdown
    (per_target.csv). Selection uses the scaled overall. Oracle mode never writes selected generator settings.
    """
    genes = read_list(a.genes, 'gene_name')
    out = Path(a.out)
    if out.exists() and any(out.iterdir()): raise ValueError(f'choose an empty benchmark directory: {out}')
    out.mkdir(parents=True, exist_ok=True)
    if a.mode == 'transfer':
        provenance = json.loads((Path(a.predictions)/'prediction_report.json').read_text())
        if provenance.get('heldout_background') != 'H1': raise ValueError('generator tuning requires the exam: predictions with H1 held out')
        if provenance.get('schema') != SCHEMA: raise ValueError('rebuild the exam predictions with vcc2_meta.py')
        if 'H1' in provenance.get('source_backgrounds', []): raise ValueError('H1 leaked into the exam sources')
    variants = benchmark_variants(a) if a.mode == 'transfer' else [dict(method='oracle', alpha=a.alpha, pi=a.pi, up_w=a.up_w, cell_rule='matched')]
    if a.mode == 'transfer':
        unknown = {v['method'] for v in variants} - set(provenance['models'])
        if unknown: raise ValueError(f'unknown models {sorted(unknown)}; the exam has {provenance["models"]}')
    if a.reference:
        # A previous stage's reference: same real cells, cache and bundle; rebuild only the generator's controls.
        ref = Path(a.reference); real_path = ref / 'real.h5ad'
        if not (ref / 'bundle').exists(): raise ValueError(f'{ref} has no competition bundle')
        targets, real_counts = reference_targets(real_path)
        split = json.loads((ref / 'control_split.json').read_text())
        if split['seed'] != a.seed: raise ValueError('reference control split uses a different seed')
        if a.n_targets and len(targets) != a.n_targets:
            raise ValueError('reference target count differs from --n-targets')
        with h5py.File(real_path) as f:
            if read_elem(f['var']).index.astype(str).tolist() != genes:
                raise ValueError('reference gene order differs from --genes')
        h = ad.read_h5ad(a.cells, backed='r')
        try: line = h1_controls(h, h.obs[a.obs_col].astype(str).values, genes, a.seed)[0]
        finally: h.file.close()
        oracle_effects = oracle_own = None
        evaluator = OfficialEvaluator(a, real_path, out, reuse=ref)
        save_json(out / 'reference_info.json', dict(reused=str(ref.resolve()), targets=len(targets)))
    else:
        real_path = out/'real.h5ad'
        line, targets, real_counts, oracle_effects, oracle_own = stream_h1(a, genes, real_path)
        evaluator = OfficialEvaluator(a, real_path, out); evaluator.prepare()
        if a.mode == 'transfer': evaluator.build_bundle()
    print(f'{len(variants)} arms to score on {len(targets)} targets', flush=True)
    if a.mode == 'transfer':
        kd = kd_residual(prediction_ontarget(a.predictions, targets), Cfg(a))
        effects = {m: (load_deltas(a.predictions, targets, genes, m), load_info(a.predictions, targets, genes, m))
                   for m in sorted({v['method'] for v in variants})}
    else:
        # Ceiling-only oracle inputs: per-cell arithmetic CPM from each target's estimation half.
        effects = {'oracle': (oracle_effects, {})}; kd = kd_residual(oracle_own, Cfg(a))
    outputs, settings, scaled, summaries = {}, {}, {}, {}
    maximum = max([a.n_cells] + list(real_counts.values()))
    pool_cfg = Cfg(a); pool_cfg.n_cells = maximum
    common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(a.seed, 'H1', 'base_controls')))
    for index, variant in enumerate(variants):
        method = variant['method']; effects_array, info = effects[method]
        label = variant.get('label') or f'{method}_{index:03d}'
        cfg = Cfg(argparse.Namespace(**dict(vars(a), **{k: v for k, v in variant.items() if k not in ('method', 'cell_rule', 'label')})))
        calibrated = calibrate(effects_array, line, targets, cfg, info=info)
        n_cells = {t: (real_counts[t] if variant['cell_rule'] == 'matched' else a.n_cells) for t in targets}
        # Normalize tuples (notably kd_clip) to their persisted JSON representation.
        effective = json.loads(json.dumps(dict(cfg.as_dict(), method=method, cell_rule=variant['cell_rule'])))
        reused = None
        if a.reference:
            previous = json.loads((Path(a.reference) / 'benchmark.json').read_text())
            for old_label, old_settings in previous['settings'].items():
                if {k: v for k, v in old_settings.items() if k != 'label'} == effective:
                    reused = Path(a.reference) / old_label
                    break
        if reused is not None:
            shutil.copytree(reused, out / label)  # scores and DE tables only; same real file, cells, knobs and seed
            save_json(out / label / 'reused_from.json', dict(arm=str(reused.resolve())))
            print(f'  reusing identical arm {reused}', flush=True)
        else:
            pred_path = out/f'{label}.h5ad'
            write_arm(pred_path, line, calibrated, kd, targets, n_cells, common, cfg)
            evaluator.score(pred_path, out/label)
            if not a.keep_arm_files: pred_path.unlink()   # ~2 GB each; the scores are in out/label
        del calibrated
        outputs[label] = str(out/label); settings[label] = dict(effective, label=label)
        save_json(out/f'{label}.json', settings[label])
        table = per_target_table(out/label); table.to_csv(out/label/'per_target.csv')
        summaries[label] = per_target_summary(table)
        if a.mode == 'transfer': scaled[label] = evaluator.scale(out/label)
        pd.DataFrame(summaries).T.to_csv(out/'per_target_summary.csv')
        if scaled:
            progress = pd.DataFrame(scaled).T.sort_values('overall', ascending=False, kind='stable')
            progress.index.name = 'arm'; progress.to_csv(out/'scaled_scores.csv')
        save_json(out / 'progress.json', dict(completed=len(outputs), total=len(variants), last_arm=label))
        print(f'official evaluation complete: {label}' + (f" (scaled overall {scaled[label]['overall']:.4f})" if label in scaled else ''), flush=True)
    pd.DataFrame(summaries).T.to_csv(out/'per_target_summary.csv')
    save_json(out/'benchmark.json', dict(mode=a.mode, outputs=outputs, settings=settings, n_targets=len(targets),
              ceiling=str(out / 'ceiling') if a.mode == 'oracle' else None,
              anchor=str((Path(a.reference) if a.reference else out) / 'anchor'),
              real_cache=str(evaluator.cache), bundle=str(evaluator.bundle),
              note='H1 is generator calibration; final evidence requires unseen lines. Oracle is a generator-only ceiling check.'))
    if a.metric_directions or a.eval_profile == 'vcc2026':
        directions = a.metric_directions or str(out/'metric_directions.csv')
        if a.metric_directions is None:
            pd.DataFrame({'metric': list(VCC2026_METRICS), 'direction': list(VCC2026_METRICS.values())}).to_csv(directions, index=False)
        frame, ranks = metric_rank_table(outputs, directions)
        frame.to_csv(out/'metric_ranks.csv', index=False); ranks.to_csv(out/'aggregate_rank.csv')
    if a.mode == 'transfer':
        table = pd.DataFrame(scaled).T.sort_values('overall', ascending=False, kind='stable')
        table.index.name = 'arm'; table.to_csv(out/'scaled_scores.csv')
        eligible = [label for label in table.index if settings[label]['cell_rule'] == 'fixed']
        if not eligible: raise ValueError('include fixed cell rule to select submission-compatible generator settings')
        best = settings[eligible[0]]
        chosen = {k: v for k, v in best.items() if k not in ('method', 'cell_rule', 'label')}
        save_json(out/'selected_generator.json', dict(settings=dict(Cfg(argparse.Namespace(**dict(vars(a), **chosen))).as_dict() | chosen, n_cells=a.n_cells),
                   cell_rule='fixed', n_cells=a.n_cells, delta_array=best['method'], arm=eligible[0],
                   scaled_overall=float(table.loc[eligible[0], 'overall']),
                   selection='scaled overall (leaderboard formula)', schema=SCHEMA,
                   note='H1 is calibration, not an untouched test'))


def cmd_tune(a):
    """Run the prescribed stages consecutively in one background process."""
    root = Path(a.out); root.mkdir(parents=True, exist_ok=True)
    submitted = json.loads(Path(a.submitted_settings).read_text())
    submitted = submitted.get('settings', submitted)
    reference = dict(method='re', call='z', sign_rule=False, detect_floor=False,
                     label='reference_v2_submitted', **{k: submitted[k] for k in ('z', 'min_effect', 'alpha', 'pi', 'up_w')})
    stage_a = [dict(method='re', call='topn', top_n=n, sign_rule=True, detect_floor=floor,
                    z_req=a.z_req, alpha=1.0, label=f'topn{n}_floor{int(floor)}')
               for n in (100, 500, 1000, 2000) for floor in (False, True)] + [reference]
    save_json(root / 'stage_a_variants.json', stage_a)
    first = copy.copy(a); first.out = str(root / 'exam_stage_a'); first.reference = None
    first.variants = str(root / 'stage_a_variants.json')
    if (Path(first.out) / 'benchmark.json').exists(): raise ValueError('stage A already exists; choose a new output root')
    cmd_benchmark(first)
    scaled = pd.read_csv(Path(first.out) / 'scaled_scores.csv', index_col='arm')
    settings = json.loads((Path(first.out) / 'benchmark.json').read_text())['settings']
    best = settings[next(label for label in scaled.index if settings[label]['call'] == 'topn')]
    stage_b = [dict(method='re', call='topn', top_n=best['top_n'], detect_floor=best['detect_floor'], z_req=best['z_req'],
                    sign_rule=sign, alpha=alpha, label=f'topn{best["top_n"]}_floor{int(best["detect_floor"])}_sign{int(sign)}_alpha{alpha:g}')
               for alpha in (1.0, 2.0, 3.0) for sign in (True, False)] + [reference]
    save_json(root / 'stage_b_variants.json', stage_b)
    second = copy.copy(a); second.out = str(root / 'exam_stage_b'); second.reference = first.out
    second.variants = str(root / 'stage_b_variants.json')
    cmd_benchmark(second)


def add_switches(p):
    p.add_argument('--alpha', type=float, default=1.0); p.add_argument('--n-cells', type=int, default=400)
    p.add_argument('--pi', type=float, default=0); p.add_argument('--up-w', type=float, default=0.75)
    p.add_argument('--metadata', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--z', type=float, default=0); p.add_argument('--min-effect', type=float, default=0)
    p.add_argument('--call', choices=['z', 'topn'], default='z'); p.add_argument('--top-n', type=int, default=1000)
    p.add_argument('--sign-rule', action='store_true'); p.add_argument('--detect-floor', action='store_true')
    p.add_argument('--z-req', type=float, default=4.0)
    p.add_argument('--match-pool', action='store_true'); p.add_argument('--tilt', action='store_true')
    p.add_argument('--seed', type=int, default=2026)


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('submit'); add_switches(s)
    for key in ['data', 'out']: s.add_argument('--'+key, required=True)
    s.add_argument('--lines', nargs='+', default=['A', 'B', 'C']); s.add_argument('--delta', action='append', required=True)
    s.add_argument('--required-n-cells', type=int, default=400)
    s.add_argument('--delta-array', default='selected'); s.add_argument('--settings')
    s.add_argument('--kd-orion', nargs='*'); s.add_argument('--kd-tcell'); s.add_argument('--kd-tcell-condition', default='Stim48hr')
    s.add_argument('--checks', type=int, default=20)
    for command, mode in [('benchmark', 'transfer'), ('oracle', 'oracle'), ('tune', 'transfer')]:
        b = sub.add_parser(command); add_switches(b); b.set_defaults(mode=mode)
        for key in ['cells', 'genes', 'out']: b.add_argument('--'+key, required=True)
        b.add_argument('--predictions', required=mode == 'transfer'); b.add_argument('--targets')
        b.add_argument('--obs-col', default='target_gene'); b.add_argument('--delta-array', nargs='+', default=['re'])
        b.add_argument('--min-cells', type=int, default=100); b.add_argument('--n-targets', type=int, default=0)
        b.add_argument('--alpha-grid', nargs='+', type=float, default=[0.75, 1.0, 1.5, 2.0])
        b.add_argument('--z-grid', nargs='+', type=float, default=[2.0, 3.0, 4.0])
        b.add_argument('--min-effect-grid', nargs='+', type=float, default=[0.1, 0.2])
        b.add_argument('--pi-grid', nargs='+', type=float, default=[0.0])
        b.add_argument('--up-w-grid', nargs='+', type=float, default=[0.75])
        b.add_argument('--cell-rules', nargs='+', choices=['matched', 'fixed'], default=['fixed'])
        b.add_argument('--cell-eval', default='cell-eval2'); b.add_argument('--eval-profile', default='vcc2026')
        b.add_argument('--num-threads', type=int, default=4)
        b.add_argument('--keep-arm-files', action='store_true', help='keep each scored arm .h5ad (deleted by default)')
        b.add_argument('--variants', help='JSON list of arms (method, call, top_n, sign_rule, detect_floor, z_req, alpha, z, min_effect, ...)')
        b.add_argument('--reference', help='a previous benchmark folder: reuse its real.h5ad, real_cache, ceiling and bundle')
        b.add_argument('--anchor-splits', type=int, default=5, help='replicate-anchor splits for the competition bundle')
        b.add_argument('--metric-directions', help='CSV metric,direction for current official scoring metrics only')
        if command == 'tune':
            b.add_argument('--submitted-settings', required=True, help='the submitted re_003.json settings to retain as reference')
    a = p.parse_args()
    if a.cmd == 'submit': cmd_submit(a)
    elif a.cmd == 'tune': cmd_tune(a)
    else: cmd_benchmark(a)


## 2. Panel sources
Rebuild `lib/sources` (QC-filtered knockdowns on the shared panel) only if it is missing, then show how many backgrounds cover each target.

In [ ]:
VOTERS = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1']   # step 1: CD4_Rest / CD4_Stim8hr do not vote
panels = [Path(ENV['SOURCES']) / s / 'effects_panel.h5ad' for s in VOTERS]
if not all(p.exists() for p in panels):
    bg('panels', r"""
    "$PY" "$SCRIPTS/knockdown_filters.py"
    "$PY" "$SCRIPTS/build_panel_sources.py"
    """)
else:
    cov = pd.read_csv(Path(ENV['SOURCES']) / 'coverage.csv', index_col='target')
    kept = cov[VOTERS].eq('kept')
    print('panel targets by number of voting backgrounds (ours = the 300 challenge targets):')
    display(pd.crosstab(kept.sum(axis=1).rename('backgrounds'), cov['panel']))

## 3. Exam
Fit τ² without H1, list H1 targets that another background also kept, and predict them with both models.

In [ ]:
run(r"""
"$PY" vcc2_meta.py fit --sources-dir "$SOURCES" --genes "$DATA/gene_names.csv" --holdout H1 --out "$WORK/models/exam"
"$PY" vcc2_meta.py exam --sources-dir "$SOURCES" --holdout H1 --out "$WORK/exam/targets.csv"
"$PY" vcc2_meta.py predict --model "$WORK/models/exam" --targets "$WORK/exam/targets.csv" --lines H1 --out "$WORK/exam/pred"
""")

Worked example from the exam model: each background's lfc, SE, weight and share for one target × gene (gene = its strongest effect), plus τ² across genes.

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_meta
EXAMPLE_TARGET, EXAMPLE_GENE = 'ACLY', None
exam_dir = Path(ENV['WORK']) / 'exam/pred/H1'
if EXAMPLE_GENE is None:
    row = list(np.load(exam_dir / 'targets.npy')).index(EXAMPLE_TARGET)
    z = np.abs(np.load(exam_dir / 're.npy', mmap_mode='r')[row]) / np.load(exam_dir / 're_se.npy', mmap_mode='r')[row]
    EXAMPLE_GENE = str(np.load(exam_dir / 'genes.npy')[np.nanargmax(np.where(np.load(exam_dir / 'n_backgrounds.npy')[row] >= 2, z, np.nan))])
frame, summary = vcc2_meta.explain(Path(ENV['WORK']) / 'models/exam', EXAMPLE_TARGET, EXAMPLE_GENE)
display(frame.round(4)); print(json.dumps(summary, indent=1))
print('tau2 quantiles (exam):', json.loads((Path(ENV['WORK']) / 'models/exam/model.json').read_text())['tau2_quantiles'])

## 4. Final model
Fit τ² on all five backgrounds and write one Δ table per challenge line.

In [ ]:
run(f"""
"$PY" vcc2_meta.py fit --sources-dir "$SOURCES" --genes "$DATA/gene_names.csv" --out "$WORK/models/final"
"$PY" vcc2_meta.py predict --model "$WORK/models/final" --targets "$DATA/pert_counts.csv" --lines {' '.join(LINES)} --out "$WORK/pred"
""")

### Future: context-aware weighting (not in this submission)
The Δ tables above are identical for A, B and C: nothing about the hidden line enters the average. Lines differ only in the generator, which uses each line's own controls (baseline, dispersion, library sizes, >5 CPM gate).
Letting the line's profile change the average itself, e.g. weighting backgrounds by how similar their controls are to line A, would be a new feature and is not part of the current spec. Plan: profile the context of each line entering the model and weight changes towards the backgrounds it most resembles, once this submission has set the baseline.

**Where it goes:** the averaging step (`combine` in `vcc2_meta.py`), not the cell generator. The generator receives a finished Δ per line; by then the background weights are used up.

**Weights:** multiply a similarity score s(line, background) into the current weight:

- now: w = 1 / (SE² + τ²)
- with similarity: w = s(line, background) / (SE² + τ²)

**SE must change with it** (it drives the DE call). √(1/Σw) only holds when w is exactly 1/(SE² + τ²). For any other weights:

SE_pred = √( Σ w² · (SE² + τ²) ) / Σ w

(With s = 1 for every background this reduces to √(1/Σw), so the current results are unchanged.)

**What stays the same:**
- τ² measures how much the sources disagree with each other, independent of the line being predicted, so `fit` is unchanged.
- `predict` already writes one folder per line; with similarity the A/B/C tables simply differ, and the generator needs no change.

**Test it on the exam first:** score H1's similarity to the four sources, rerun section 3 and the exam benchmark, and compare against this submission's baseline before using it for A/B/C.

**The hard part** is the profile itself: a similarity score that is comparable across sources measured on different platforms (Orion pseudobulk, Replogle K562, T-cell pseudobulk, H1 Flex).

## 5. Tune on the exam
Stage A: top-N calling with the sign rule, N ∈ {100, 500, 1000, 2000} × detection floor off / on at α = 1, plus the submitted settings as a reference. Stage B runs automatically after A: the best top-N arm × α {1, 2, 3} × sign rule on/off, plus the same reference.

Use 50 targets while local disk space is tight (12 GB free at setup); set 75 when capacity permits. Both stages use the same real cells, seed and official competition bundle. Selection uses the six scaled leaderboard metrics' mean. The CLI default is z_req = 4; this run uses 5, which passed the five-knockdown detection check.

The installed cell-eval2 requires `score --real-bundle` to report the official overall. `score --baseline-agg ... --anchor ...` reports diagnostic scores only. The bundle binds the generic baseline and cached `run --anchor` result to the same real exam.

In [ ]:
BENCHMARK_TARGETS = 50   # allowed fallback for tight local disk; use 75 when capacity permits
EXAM = (f'--cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/exam/targets.csv" --predictions "$WORK/exam/pred/H1" '
        f'--eval-profile vcc2026 --n-targets {BENCHMARK_TARGETS} --anchor-splits 5 --num-threads 2 --seed 2026 --z-req 5')
# One background job runs both stages. Every completed arm updates the scaled table and progress.json.
bg('tune', f'OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 "$PY" vcc2_generate.py tune {EXAM} --submitted-settings "$LIB/vcc_2026_2/exam_b/re_003.json" --out "$WORK"')

Stage A progress, then each arm's scaled scores (leaderboard formula, sorted by overall), per-knockdown medians and raw metrics.

In [ ]:
def arms(folder):
    folder = Path(ENV['WORK']) / folder
    scaled = pd.read_csv(folder / 'scaled_scores.csv', index_col='arm')
    settings = pd.DataFrame({label: json.loads((folder / f'{label}.json').read_text()) for label in scaled.index}).T
    per_target = pd.read_csv(folder / 'per_target_summary.csv', index_col=0)
    raw_path = folder / 'metric_ranks.csv'
    raw = (pd.read_csv(raw_path).pivot(index='variant', columns='metric', values='value').add_prefix('raw_')
           if raw_path.exists() else pd.DataFrame(index=scaled.index))
    return settings.join(scaled.add_prefix('scaled_')).join(per_target).join(raw).sort_values('scaled_overall', ascending=False)
status('tune', n=5)
if (Path(ENV['WORK']) / 'exam_stage_a/scaled_scores.csv').exists(): display(arms('exam_stage_a'))

Stage B starts automatically after stage A. It scores the best top-N arm's N and floor × α {1, 2, 3} × sign rule on/off, plus the submitted reference. Identical arms reuse stage A's saved scores and DE tables; every arm uses the same real file, control split and competition bundle.

In [ ]:
variants_b = Path(ENV['WORK']) / 'stage_b_variants.json'
if variants_b.exists():
    display(pd.DataFrame(json.loads(variants_b.read_text())))
else:
    print('Stage B will be planned and started automatically when stage A finishes.')

Stage B progress, arms and the selected settings: call mode, N, floor, sign rule, z_req and α (these drive generation).

In [ ]:
status('tune', n=5)
settings = Path(ENV['WORK']) / 'exam_stage_b/selected_generator.json'
if settings.exists():
    display(arms('exam_stage_b')); print(settings.read_text())

Optional: generator ceiling on H1 using effects estimated from half of each target's own cells; never selects settings.

In [ ]:
RUN_ORACLE = False
if RUN_ORACLE:
    bg('oracle_h1', f'"$PY" vcc2_generate.py oracle --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/exam/targets.csv" --eval-profile vcc2026 --n-targets {BENCHMARK_TARGETS} --out "$WORK/oracle_h1"')

### Future: K562 as a second exam line (not run)
- **Why:** H1 is one unusual line (embryonic stem cells, a very strong shared response), while A–C are cancer lines. K562 has raw cells and genome-wide coverage.
- **What it needs:** hold K562 out of `fit` (`--holdout K562`, which also removes it from τ²); build its exam targets and real-cell reference and cache from the Replogle file; repeat stages A / B; choose settings that do well on both lines (or best on the worse one).
- **Cost:** roughly doubles exam compute.
- **Trigger:** use it if leaderboard results for A–C diverge from what the H1 exam predicts.

## 6. Generate
Cells for every challenge line from its Δ table with the stage-B settings (call mode, N, floor, sign rule, α); refuses to overwrite a finished run, so use a new `RUN_NAME` each time.

In [ ]:
settings = Path(ENV['WORK']) / 'exam_stage_b/selected_generator.json'
if not settings.exists(): raise FileNotFoundError('Finish exam stage B before generation.')
result = V / 'results' / RUN_NAME
if result.exists() and any(result.iterdir()): raise FileExistsError('Choose a new RUN_NAME to preserve existing results.')
line_args = ' '.join(shlex.quote(line) for line in LINES)
delta_args = ' '.join(f'--delta {shlex.quote(line + "=" + str(Path(ENV["WORK"]) / "pred" / line))}' for line in LINES)
bg('generate', f'"$PY" vcc2_generate.py submit --data "$DATA" --lines {line_args} --delta-array re --settings {shlex.quote(str(settings))} {delta_args} --out {shlex.quote(str(result))}')

In [ ]:
status('generate')

## 7. Package
`vcc prep` validates and packages the run. It holds the whole matrix in memory (~30 GB at peak for 360k cells), so it stops early unless RAM + swap is enough; leaderboard submission and bucket backup stay off until switched on.

In [ ]:
SUBMISSION = V / 'results' / RUN_NAME
memory = {k: int(v.split()[0]) / 1e6 for k, v in (line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()) if k in ('MemAvailable', 'SwapFree')}
if memory['MemAvailable'] + memory['SwapFree'] < 34:
    raise RuntimeError(f"only {memory['MemAvailable']:.0f} GB RAM + {memory['SwapFree']:.0f} GB swap free; vcc prep peaks near 30 GB. "
                       'Add swap first: sudo fallocate -l 12G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile')
run(f'vcc prep {shlex.quote(str(SUBMISSION / "prediction.h5ad"))} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {shlex.quote(str(SUBMISSION / "prediction.vcc"))}')

In [ ]:
SUBMIT_TO_LEADERBOARD = False
MODEL_NAME = 'LHY'   # shown publicly on the leaderboard: keep it free of method details
if SUBMIT_TO_LEADERBOARD:
    chosen = json.loads((Path(ENV['WORK']) / 'exam_stage_b/selected_generator.json').read_text())
    s = chosen['settings']
    print(f"submitting as {MODEL_NAME!r}; private record: {chosen['arm']}: call={s['call']}, N={s.get('top_n')}, floor={s.get('detect_floor')}, "
          f"sign={s.get('sign_rule')}, alpha={s['alpha']}, exam scaled overall={chosen['scaled_overall']:.4f}")
    run(f'vcc submit {shlex.quote(str(SUBMISSION / "prediction.vcc"))} -m {shlex.quote(MODEL_NAME)} --wait')

In [ ]:
BACK_UP_TO_BUCKET = False
if BACK_UP_TO_BUCKET:
    run(r"""
    gcloud storage rsync -r "$WORK" "$B/lib/vcc_2026_2a" --exclude='.*\.h5ad$'
    gcloud storage rsync -r "$CODE" "$B/code/vcc_2026_2a"
    gcloud storage cp ~/vcc/vcc_2026_2a.ipynb "$B/code/vcc_2026_2a.ipynb"
    """)

Submission plan: after stage B, submit **one** leaderboard entry with the selected settings. If the daily allowance permits, submit a second with a different N (rerun generation under a new `RUN_NAME` with that N), to check that A–C respond to call count the way H1 does — their real DE counts may differ from H1's ~2,020 per knockdown.

## Ad hoc tests
Checks for this notebook's edits (1 and 5) and the generic-response questions (D5, D6, edits 8–10). Nothing here changes a submission; outputs go to `lib/vcc_2026_2a/`.

Write the diagnostics script (vcc_2026_2's D1–D4 plus D5, D6 and the detection-floor check).

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2a/vcc2_adhoc.py
"""Ad hoc diagnostics after the vcc_2026_2 submission (step 0 of the review). Nothing here changes the submission.

  arms  regenerate four arms on the exam's 25 H1 targets, reusing the exam's real file and real-side cache, and score
        each with cell-eval2, keeping both sides' DE tables (--write-degenes):
          null_shared       delta = 0 and no knockdown; every target is the one shared 400-cell pool (pure pool artefact)
          null_independent  delta = 0 and no knockdown; a fresh 400-cell draw per target
          submitted         the submitted settings (exam_b's selection)
          z2                z* = 2, min 0.1, alpha 1 (stage A's best arm on fid and Jaccard)
        The shared-pool arms rebuild the benchmark's cells exactly (same control split, pool and seeds), so 'submitted'
        must score as its exam_b arm did.

Functions used by the notebook cells:
  de_table(arm)        per-target predicted vs real DE genes (scorer's p_adj < 0.05, own gene excluded)
  generic_share(...)   how much of H1's real DE lands on genes DE in > 30% of knockdowns
  generic_transfer()   correlation of each source's average response with H1's
"""
from __future__ import annotations

import argparse
import json
import subprocess
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
from scipy import stats

import vcc2_generate as g
from vcc2_common import alignment, expression_gate, moments, seed_for, save_json

try:
    from anndata.io import read_elem
except ImportError:  # older anndata
    from anndata.experimental import read_elem

CONTROL = 'non-targeting'
SEED = 2026
ARMS = ['null_shared', 'null_independent', 'submitted', 'z2']
SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr']


# ---------------------------------------------------------------- arms
def exam_line(cells, genes, seed=SEED):
    """The benchmark's generator controls and its scoring-half gate, rebuilt as stream_h1 does."""
    h = ad.read_h5ad(cells, backed='r')
    try:
        labels = h.obs['target_gene'].astype(str).values
        ci = np.random.default_rng(seed_for(seed, 'H1', 'controls')).permutation(np.flatnonzero(labels == CONTROL))
        train, score = ci[:len(ci) // 2], ci[len(ci) // 2:]
        small = h[np.sort(train)].to_memory()
        line = g.Line(g.aligned_X(small, genes), None, genes); del small
        line.gate = expression_gate(moments(h, np.sort(score), alignment(h.var_names.astype(str), genes))['mean_cpm'])
    finally:
        h.file.close()
    return line


def exam_targets(real):
    a = ad.read_h5ad(real, backed='r')
    try: counts = a.obs['target_gene'].astype(str).value_counts()
    finally: a.file.close()
    counts = counts.drop(CONTROL)
    return sorted(counts.index), counts.to_dict()


def write_arm(path, line, targets, effects, kd, picks, cfg, own):
    labels = [t for t in targets for _ in range(cfg.n_cells)] + [CONTROL] * line.n
    writer = g.StreamingH5AD(path, g.labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            writer.append(g.generate_one(line, effects[i], own[i], None if kd is None else kd[i], cfg,
                                         pick=picks[i], line_key='H1', target_key=t)[0])
        for start in range(0, line.n, 4096):
            writer.append(line.X[start:start + 4096].astype(np.int32))
    finally:
        writer.close()


def score(binary, pred, real, cache, out, threads=4):
    command = [binary, 'run', '-ap', pred, '-ar', real, '-o', out, '--cache-real', cache, '--pert-col', 'target_gene',
               '--control', CONTROL, '--profile', 'vcc2026', '--input-type', 'counts', '--preset', 'vcc2026',
               '--set', 'de.backend=pdex', '--set', f'num_threads={threads}', '--write-degenes']
    print('  $ ' + ' '.join(map(str, command)), flush=True)
    subprocess.run([str(x) for x in command], check=True)


def cmd_arms(a):
    genes = g.read_list(a.genes, 'gene_name'); exam = Path(a.exam); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    targets, real_counts = exam_targets(exam / 'real.h5ad')
    print(f'{len(targets)} exam targets; rebuilding the generator controls from {a.cells}', flush=True)
    line = exam_line(a.cells, genes)
    selected = json.loads((exam / 'selected_generator.json').read_text())['settings']
    settings = dict(submitted=dict(alpha=selected['alpha'], z=selected['z'], min_effect=selected['min_effect']),
                    z2=dict(alpha=1.0, z=2.0, min_effect=0.1))
    base = dict(pi=0.0, up_w=0.75, n_cells=a.n_cells, seed=SEED)
    pool_cfg = g.Cfg(argparse.Namespace(**base)); pool_cfg.n_cells = max([a.n_cells] + list(real_counts.values()))
    common = g.pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(SEED, 'H1', 'base_controls')))
    shared = [common[:a.n_cells]] * len(targets)
    cfg0 = g.Cfg(argparse.Namespace(**base))
    independent = [g.pick_cells(line, np.zeros(len(genes)), cfg0, np.random.default_rng(seed_for(SEED, 'H1', 'independent', t)))
                   for t in targets]
    gp = {x: i for i, x in enumerate(genes)}; own = [gp.get(t) for t in targets]
    zeros = np.zeros((len(targets), len(genes)), np.float32)
    D = g.load_deltas(a.predictions, targets, genes, 're'); se = g.load_deltas(a.predictions, targets, genes, 're_se', fill=np.inf)
    plan = dict(null_shared=(zeros, None, shared, [None] * len(targets), cfg0),
                null_independent=(zeros, None, independent, [None] * len(targets), cfg0))
    for name, s in settings.items():
        cfg = g.Cfg(argparse.Namespace(**base, **s))
        plan[name] = (g.calibrate(D, line, targets, cfg, se), g.kd_residual(g.prediction_ontarget(a.predictions, targets), cfg),
                      shared, own, cfg)
    for name in a.arms:
        effects, kd, picks, owns, cfg = plan[name]
        path = out / f'{name}.h5ad'
        write_arm(path, line, targets, effects, kd, picks, cfg, owns)
        score(a.cell_eval, path, exam / 'real.h5ad', exam / 'real_cache', out / name, a.num_threads)
        path.unlink()                                             # ~2 GB; the scores and DE tables are kept
        save_json(out / name / 'arm.json', dict(arm=name, settings={k: v for k, v in vars(cfg).items() if k in ['alpha', 'z', 'min_effect']},
                                                pool='independent' if name == 'null_independent' else 'shared', targets=targets))
        print(f'arm complete: {name}', flush=True)


# ---------------------------------------------------------------- analysis
def de_table(arm_dir, alpha=0.05):
    """Per target: predicted and real DE genes (p_adj < alpha, own gene excluded), overlap, precision, recall, and the
    chance levels: random genes would hit real DE genes at n_real / scored genes, and get the direction right half the time."""
    pred = pd.read_parquet(Path(arm_dir) / 'de_pred.parquet'); real = pd.read_parquet(Path(arm_dir) / 'de_real.parquet')
    rows = []
    for t, r in real.groupby('target'):
        p = pred[pred.target == t]
        P = set(p.feature[(p.p_adj < alpha) & (p.feature != t)]); R = set(r.feature[(r.p_adj < alpha) & (r.feature != t)])
        both = P & R
        same_sign = 0
        if both:
            lp = p.set_index('feature').log2_fold_change; lr = r.set_index('feature').log2_fold_change
            same_sign = int(sum(np.sign(lp[x]) == np.sign(lr[x]) for x in both))
        scored = r.feature.nunique()
        precision = len(both) / len(P) if P else np.nan
        rows.append(dict(target=t, n_pred=len(P), n_real=len(R), overlap=len(both), same_direction=same_sign,
                         precision=precision, chance_precision=len(R) / scored, enrichment=precision * scored / len(R) if R else np.nan,
                         direction_agreement=same_sign / len(both) if both else np.nan, recall=len(both) / len(R) if R else np.nan))
    return pd.DataFrame(rows).set_index('target')


def generic_from_de(de, alpha=0.05, share=0.3):
    """From a scorer DE table: genes DE in > `share` of targets, and the fraction of all DE calls on them."""
    calls = de[(de.p_adj < alpha) & (de.feature != de.target)]
    n_targets = de.target.nunique()
    freq = calls.groupby('feature').target.nunique() / n_targets
    generic = set(freq.index[freq > share])
    return dict(targets=n_targets, de_calls=len(calls), generic_genes=len(generic),
                calls_on_generic=int(calls.feature.isin(generic).sum()),
                share_of_calls_on_generic=float(calls.feature.isin(generic).mean()) if len(calls) else np.nan), generic, freq


def effect_table(path, min_cpm):
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        keep = np.flatnonzero(var['ctrl_cpm'].to_numpy(float) > min_cpm)
        lfc = f['X'][:][:, keep]; se = f['layers/se'][:][:, keep]
    genes = var.index.astype(str).to_numpy()[keep]; targets = obs.index.astype(str).to_numpy()
    gp = {x: i for i, x in enumerate(genes)}
    for i, t in enumerate(targets):                                   # own gene is not a downstream response
        if t in gp: lfc[i, gp[t]] = np.nan
    return targets, genes, lfc, se


def generic_from_effects(path, alpha=0.05, share=0.3, min_cpm=5.0):
    """Same question on a pseudobulk effect table: z = lfc / SE, BH within each knockdown (like the scorer's per-pert FDR)."""
    targets, genes, lfc, se = effect_table(path, min_cpm)
    with np.errstate(invalid='ignore', divide='ignore'):
        p = 2 * stats.norm.sf(np.abs(lfc / se))
    de = np.zeros_like(lfc, bool)
    for i in range(len(targets)):
        ok = np.isfinite(p[i])
        if ok.any(): de[i, np.flatnonzero(ok)] = stats.false_discovery_control(p[i, ok]) < alpha
    freq = de.mean(0); generic = freq > share
    calls = int(de.sum())
    return dict(targets=len(targets), genes=len(genes), de_calls=calls, generic_genes=int(generic.sum()),
                calls_on_generic=int(de[:, generic].sum()),
                share_of_calls_on_generic=float(de[:, generic].sum() / calls) if calls else np.nan), pd.Series(freq, index=genes)


def average_response(path, min_cpm=1.0):
    targets, genes, lfc, _ = effect_table(path, min_cpm)
    return pd.Series(np.nanmean(lfc, 0), index=genes), len(targets)


def generic_transfer(filters_dir, h1_generic_freq, real_de=None, top=200):
    """Each source's average knockdown response vs H1's: Pearson r, and overlap of the strongest genes."""
    h1, n_h1 = average_response(Path(filters_dir) / 'H1.h5ad', min_cpm=5.0)
    references = {'H1 effect table (all kept knockdowns)': h1}
    if real_de is not None:
        references['H1 scorer DE (25 exam targets)'] = real_de[real_de.feature != real_de.target].groupby('feature').log2_fold_change.mean()
    sources = {name: average_response(Path(filters_dir) / f'{name}.h5ad')[0] for name in SOURCES}
    sources['mean of the 4 sources'] = pd.concat(sources.values(), axis=1).mean(axis=1)
    generic = set(h1_generic_freq.index[h1_generic_freq > 0.3])
    rows = []
    for ref_name, ref in references.items():
        for name, s in sources.items():
            common = ref.index.intersection(s.index); x, y = s[common], ref[common]
            ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
            top_ref = set(y.abs().nlargest(top).index); top_src = set(x.abs().nlargest(top).index)
            src_on_generic = set(x.abs().nlargest(len(generic)).index) & generic if generic else set()
            rows.append(dict(reference=ref_name, source=name, genes=len(x), pearson=float(np.corrcoef(x, y)[0, 1]),
                             spearman=float(stats.spearmanr(x, y).statistic), **{f'top{top}_overlap': len(top_ref & top_src) / top},
                             same_sign_on_top=float(np.mean(np.sign(x[list(top_ref & top_src)]) == np.sign(y[list(top_ref & top_src)])))
                             if top_ref & top_src else np.nan,
                             h1_generic_recovered=len(src_on_generic) / len(generic) if generic else np.nan,
                             source_mean_abs=float(x.abs().mean()), h1_mean_abs=float(y.abs().mean())))
    return pd.DataFrame(rows).set_index(['reference', 'source'])


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('arms')
    for key in ['cells', 'genes', 'exam', 'predictions', 'out']: s.add_argument('--' + key, required=True)
    s.add_argument('--arms', nargs='+', default=ARMS, choices=ARMS); s.add_argument('--n-cells', type=int, default=400)
    s.add_argument('--cell-eval', default='cell-eval2'); s.add_argument('--num-threads', type=int, default=4)
    a = p.parse_args()
    cmd_arms(a)


# ---------------------------------------------------------------- D5 / D6: where could a line's generic response come from?
LIB = Path.home() / 'vcc' / 'lib'
STRESS = "HSPA1A HSPA1B HSPA8 HSPH1 DNAJB1 HSP90AA1 FOS JUN EGR1 ATF3 IER2 DUSP1 DDIT3 ATF4".split()
CC_S, CC_G2M = g.CC_GENES[:g.CC_GENES.index('HMGB2')], g.CC_GENES[g.CC_GENES.index('HMGB2'):]
CONTROL_CELLS = dict(H1=(LIB / 'h1/H1_full_2025.h5ad', 'target_gene', None),           # single-cell controls on disk
                     K562_gwps=(LIB / 'replogle/K562_gwps_raw_singlecell_01.h5ad', 'gene', 'UMI_count'),
                     HCT116=(LIB / 'orion/HCT116_cells_targets.h5ad', 'target_gene', None),
                     HEK293T=(LIB / 'orion/HEK293T_cells_targets.h5ad', 'target_gene', None))


def control_logcpm(name, n_cells=4000, seed=SEED):
    """log1p(per-cell CPM) of a random sample of the line's own control cells (library = all its genes, or the
    file's UMI total where the matrix holds only some genes, as for Replogle)."""
    path, col, library_col = CONTROL_CELLS[name]
    a = ad.read_h5ad(path, backed='r')
    try:
        rows = np.flatnonzero(a.obs[col].astype(str).values == CONTROL)
        rows = np.sort(np.random.default_rng(seed_for(seed, 'D5', name)).choice(rows, min(n_cells, len(rows)), replace=False))
        genes = a.var['gene_name'].astype(str).to_numpy() if 'gene_name' in a.var else a.var_names.astype(str).to_numpy()
        lib = a.obs[library_col].to_numpy(float)[rows] if library_col else None
    finally:
        a.file.close()
    with h5py.File(path, 'r') as f:
        if isinstance(f['X'], h5py.Dataset):
            X = f['X'][rows].astype(np.float32)
        else:
            a = ad.read_h5ad(path, backed='r')
            try: X = a[rows].to_memory().X
            finally: a.file.close()
            X = X.toarray().astype(np.float32) if hasattr(X, 'toarray') else np.asarray(X, np.float32)
    if lib is None: lib = X.sum(1)
    return genes, np.log1p(X / np.maximum(lib, 1)[:, None] * 1e6)


def control_axes(L, genes, k=20):
    """Gene-space directions of variation among the controls: top-k PC loadings, and cell-cycle / stress axes
    (mean log expression in the top minus the bottom quartile of control cells scored on that program)."""
    from sklearn.utils.extmath import randomized_svd
    Z = L - L.mean(0)
    _, _, vt = randomized_svd(Z, n_components=k, n_iter=5, random_state=SEED)
    gp = {x: i for i, x in enumerate(genes)}; axes = {}
    for label, members in [('cycle_S', CC_S), ('cycle_G2M', CC_G2M), ('stress', STRESS)]:
        idx = [gp[x] for x in members if x in gp]
        if len(idx) < 3: continue
        score = L[:, idx].mean(1); lo, hi = np.quantile(score, [0.25, 0.75])
        axes[label] = L[score >= hi].mean(0) - L[score <= lo].mean(0)
    return pd.DataFrame(vt.T, index=genes, columns=[f'PC{i + 1}' for i in range(k)]), pd.DataFrame(axes, index=genes)


def r2_with_cv(y, X, folds=5, seed=SEED):
    """OLS R^2 of a gene vector on gene-level predictors, in-sample and with folds over GENES (fit on some genes,
    predict the others), so a direction learned on one gene set must hold on another."""
    y = np.asarray(y, float); X = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    fit = lambda rows: np.linalg.lstsq(X[rows], y[rows], rcond=None)[0]
    tss = ((y - y.mean()) ** 2).sum()
    if len(y) < folds or tss == 0: return np.nan, np.nan
    in_sample = 1 - ((y - X @ fit(np.arange(len(y)))) ** 2).sum() / tss
    fold = np.empty(len(y), int)
    fold[np.random.default_rng(seed).permutation(len(y))] = np.arange(len(y)) % folds
    pred = np.empty(len(y))
    for f in range(folds):
        pred[fold == f] = X[fold == f] @ fit(np.flatnonzero(fold != f))
    return float(in_sample), float(1 - ((y - pred) ** 2).sum() / tss)


def d5(lines=('H1', 'K562_gwps', 'HCT116', 'HEK293T'), n_cells=4000, k=20, output=None):
    """Is a line's generic response (mean lfc over its kept knockdowns, genes > 5 CPM) a direction visible in its
    own controls? R^2 on the top-k control PCs, on the cycle / stress axes, and on both, in-sample and gene-CV."""
    rows = []
    for name in lines:
        print(f'D5: {name}, up to {n_cells} control cells', flush=True)
        generic, n_kd = average_response(LIB / 'filters' / f'{name}.h5ad', min_cpm=5.0)
        genes, L = control_logcpm(name, n_cells)
        keep = np.isin(genes, generic.index[np.isfinite(generic.to_numpy())])
        _, first = np.unique(genes, return_index=True); keep &= np.isin(np.arange(len(genes)), first)
        genes, L = genes[keep], L[:, keep]
        pcs, axes = control_axes(L, genes, k)
        y = generic.reindex(genes).to_numpy()
        for label, X in [(f'top {k} control PCs', pcs), ('cycle + stress axes', axes), ('both', pd.concat([pcs, axes], axis=1))]:
            r2, cv = r2_with_cv(y, X.to_numpy())
            rows.append(dict(line=name, predictors=label, n_predictors=X.shape[1], genes=len(y), knockdowns=n_kd,
                             control_cells=L.shape[0], r2=r2, r2_gene_cv=cv))
    frame = pd.DataFrame(rows).set_index(['line', 'predictors'])
    if output is not None:
        Path(output).parent.mkdir(parents=True, exist_ok=True); frame.to_csv(output)
    return frame


def d6(names=('HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1'), cancer=('HCT116', 'HEK293T', 'K562_gwps'), output=None):
    """Generic response (mean lfc over each source's kept knockdowns): pairwise r on shared measured genes, and each
    line predicted from the average of the others (all others, and the cancer lines only)."""
    G = {n: average_response(LIB / 'filters' / f'{n}.h5ad', min_cpm=1.0)[0] for n in names}
    def r(x, y):
        common = x.index.intersection(y.index); a_, b_ = x[common], y[common]; ok = np.isfinite(a_) & np.isfinite(b_)
        return float(np.corrcoef(a_[ok], b_[ok])[0, 1])
    pairwise = pd.DataFrame({x: {y: r(G[x], G[y]) for y in names} for x in names})
    loo = pd.DataFrame({n: dict(all_others=r(G[n], pd.concat([G[o] for o in names if o != n], axis=1).mean(axis=1)),
                                cancer_lines=r(G[n], pd.concat([G[o] for o in cancer if o != n], axis=1).mean(axis=1)))
                        for n in names}).T
    if output is not None:
        out = Path(output); out.mkdir(parents=True, exist_ok=True)
        pairwise.to_csv(out / 'd6_pairwise.csv'); loo.to_csv(out / 'd6_leave_one_out.csv')
    return pairwise, loo


# ---------------------------------------------------------------- edit 5 check: do the genes we call show up as DE?
def per_cell_cpm(X):
    X = X.tocsr().astype(np.float64)
    lib = np.asarray(X.sum(1)).ravel()
    return X.multiply(1e6 / np.maximum(lib, 1)[:, None]).tocsc()


def wilcoxon_blocks(pred, real, block=256):
    """Independent gene tests in small blocks; apply BH once to the returned full vector."""
    p = np.empty(pred.shape[1])
    for start in range(0, len(p), block):
        stop = min(start + block, len(p))
        p[start:stop] = stats.mannwhitneyu(pred[:, start:stop].toarray(), real[:, start:stop].toarray(), axis=0).pvalue
    return p


def floor_check(predictions, cells, genes_csv, n_targets=5, top_n=1000, z_req=4.0, sign_rule=True, alpha=1.0, seed=SEED):
    """Generate ~5 exam knockdowns with the top-N call, without and with the detection floor, and test every gene
    > 5 CPM in the generated 400 cells against the scorer's half of the H1 controls (Mann-Whitney, BH within the
    knockdown, as the scorer's per-perturbation FDR). Reports the share of CALLED genes that come out significant
    and in the called direction."""
    genes = g.read_list(genes_csv, 'gene_name')
    h = ad.read_h5ad(cells, backed='r')
    try:
        labels = h.obs['target_gene'].astype(str).values
        line, _, score_rows = g.h1_controls(h, labels, genes, seed)
        ref = per_cell_cpm(g.aligned_X(h[np.sort(score_rows)].to_memory(), genes))
    finally:
        h.file.close()
    available = list(np.load(Path(predictions) / 'targets.npy').astype(str))
    targets = sorted(np.random.default_rng(seed_for(seed, 'floor_check')).choice(available, n_targets, replace=False).tolist())
    gated = np.flatnonzero(line.gate); R = ref[:, gated]
    ref_mean = np.asarray(R.mean(0)).ravel()
    gp = {x: i for i, x in enumerate(genes)}
    rows = []
    for floor in (False, True):
        cfg = g.Cfg(argparse.Namespace(call='topn', top_n=top_n, sign_rule=sign_rule, detect_floor=floor, z_req=z_req, alpha=alpha, seed=seed))
        D, info = g.load_effects(predictions, targets, genes, 're', cfg)
        C = g.calibrate(D, line, targets, cfg, info=info)
        kd = g.kd_residual(g.prediction_ontarget(predictions, targets), cfg)
        pick = g.pick_cells(line, np.zeros(len(genes)), cfg, np.random.default_rng(seed_for(seed, 'H1', 'base_controls')))
        for i, t in enumerate(targets):
            cells_t = g.generate_one(line, C[i], gp.get(t), kd[i], cfg, pick=pick, line_key='H1', target_key=t)[0]
            P = per_cell_cpm(cells_t)[:, gated]
            p = wilcoxon_blocks(P, R)
            padj = np.ones_like(p); ok = np.isfinite(p); padj[ok] = stats.false_discovery_control(p[ok])
            called = np.flatnonzero(C[i, gated] != 0)
            sig = padj[called] < 0.05
            pred_mean = np.asarray(P.mean(0)).ravel()
            direction = np.sign(pred_mean[called] - ref_mean[called]) == np.sign(C[i, gated][called])
            rows.append(dict(target=t, detect_floor=floor, n_called=len(called), significant=float(sig.mean()) if len(called) else np.nan,
                             significant_right_direction=float((sig & direction).mean()) if len(called) else np.nan,
                             uncalled_significant=int((padj < 0.05).sum() - sig.sum()),
                             median_abs_effect=float(np.median(np.abs(C[i, gated][called]))) if len(called) else np.nan))
            print(f'floor={floor}, {t}: {len(called)} calls, {sig.mean():.1%} significant', flush=True)
    return pd.DataFrame(rows).set_index(['detect_floor', 'target'])

Edit 1 check, step 1 (background, ~1 h, only if not done yet): build the competition bundle on `vcc_2026_2`'s 25-target exam reference and score all of that notebook's stage A / B arms against it.

In [ ]:
RESCORE = Path(ENV['WORK']) / 'rescore_v2'
if not (RESCORE / 'exam_b_re_003.csv').exists():
    bg('rescore_v2', r"""
C=$(dirname "$PY")/cell-eval2; X="$LIB/vcc_2026_2/exam_b"; O="$WORK/rescore_v2"
COMMON="--pert-col target_gene --control non-targeting --profile vcc2026 --input-type counts --preset vcc2026 --set de.backend=pdex --set num_threads=4"
$C baseline -ar "$X/real.h5ad" -o "$O/baseline" --save-pred "$O/baseline/baseline_pred.h5ad" --cache-real "$X/real_cache" $COMMON
$C prep-real-bundle --real "$X/real.h5ad" --baseline "$O/baseline/baseline_pred.h5ad" -o "$O/bundle" --cache-real "$X/real_cache" $COMMON
rm -f "$O/baseline/baseline_pred.h5ad"
for stage in exam_a exam_b; do for arm in "$LIB/vcc_2026_2/$stage"/re_0*/; do
  $C score --user-agg "$arm/agg_results.csv" --real-bundle "$O/bundle" --user-meta "$arm/run_meta.json" -o "$O/${stage}_$(basename "$arm").csv"
done; done
""")
else:
    print('already scored:', RESCORE)

Edit 1 check, step 2: every `vcc_2026_2` exam arm on the leaderboard scale. `exam_b re_003` was submitted; done when it shows the leaderboard pattern (fid < 0, mse = 0, pds > 0). Last column: which arm the leaderboard formula would have chosen.

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_generate
RESCORE = Path(ENV['WORK']) / 'rescore_v2'
if (RESCORE / 'exam_b_re_003.csv').exists():
    rows = {}
    for path in sorted(RESCORE.glob('exam_*_re_*.csv')):
        stage, arm = path.stem.split('_re_'); arm = 're_' + arm
        settings = json.loads((V / 'lib/vcc_2026_2' / stage / f'{arm}.json').read_text())
        rows[f'{stage} {arm}'] = dict(z=settings['z'], min_effect=settings['min_effect'], alpha=settings['alpha'], **vcc2_generate.read_scaled(path))
    rescored = pd.DataFrame(rows).T.sort_values('overall', ascending=False)
    display(rescored.round(3))
    check = json.loads((RESCORE / 're_003/check.json').read_text()) if (RESCORE / 're_003/check.json').exists() else None
    if check:
        display(pd.Series(check['medians'], name='submitted re_003'))
    s = rescored.loc['exam_b re_003']
    print('re_003 shows the leaderboard pattern (fid < 0, mse = 0, pds > 0):',
          bool(s['de_wilcoxon_direction_fidelity_yield_raw'] < 0 and s['expr_mse_unbiased_capped_norm'] == 0 and s['pds_cosine'] > 0))
else:
    status('rescore_v2', n=5)

Edit 5 check: share of called genes the generated 400 cells show as DE (local Wilcoxon against the scorer's control half, BH within each knockdown) on 5 exam knockdowns, top 1,000 with the sign rule, without and with the detection floor. Done when ≥ 80% with the floor.

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_adhoc
floor = vcc2_adhoc.floor_check(Path(ENV['WORK']) / 'exam/pred/H1', H1_CELLS, Path(ENV['DATA']) / 'gene_names.csv', z_req=5.0)
floor.to_csv(Path(ENV['WORK']) / 'floor_check_zreq5.csv')
display(floor.round(3))
summary = floor.groupby(level='detect_floor')[['n_called', 'significant', 'significant_right_direction', 'median_abs_effect']].mean()
display(summary.round(3))
assert floor.xs(True).significant.ge(0.8).all(), 'Fewer than 80% significant for at least one knockdown'
print('Detection floor passed: at least 80% of calls significant for every checked knockdown.')

The CLI keeps the requested approximate z_req = 4 default. The saved earlier check averaged 79.7% significant at 4; z_req = 5 gave 91.9–93.9% across all five knockdowns. The benchmark and the check above therefore use 5 explicitly.

In [ ]:
zreq5 = pd.read_csv(Path(ENV['WORK']) / 'floor_check_zreq5.csv', index_col=['detect_floor', 'target'])
summary5 = zreq5.groupby(level='detect_floor')[['n_called', 'significant', 'significant_right_direction', 'median_abs_effect']].mean()
display(summary5.round(3))
print('>= 80% of called genes significant with the floor at z_req = 5:', bool(summary5.loc[True, 'significant'] >= 0.8))

D5: regress each line's generic response on its own top 20 control PCs, cell-cycle/stress axes, and both. Report in-sample and gene-split cross-validated R² for H1, K562, HCT116 and HEK293T. Save the table as `d5_control_r2.csv`.

In [ ]:
d5 = vcc2_adhoc.d5(output=Path(ENV['WORK']) / 'd5_control_r2.csv')
display(d5.round(3))

D6 (edit 9): does the shared response transfer between lines? Pairwise r of each source's generic vector, and each line predicted from the average of the others (all, and the cancer lines only).

In [ ]:
pairwise, loo = vcc2_adhoc.d6(output=ENV['WORK'])
display(pairwise.round(2)); display(loo.round(2))

### Decision gate (edit 10): generic + specific split — not built for now
Recorded 2026-10-12 from the D5 / D6 outputs above.
- **D5:** H1's generic response lies in the span of its own control PCs (gene-CV R² 0.35; K562 0.42), and it is not an expression-level artefact (expression level alone: 0.04 / 0.08). It passes the ≥ 0.3 gate numerically but not its purpose: the fit needs the line's own knockdown response, the R² is spread over many PCs (none above r² 0.09) with line-specific signs, and a rule built from features any line's controls provide does not transfer (trained on H1 → K562 r = -0.05; K562 → H1 r = 0.22). It cannot give A–F a generic direction from their controls.
- **D6:** cancer-line leave-one-out r is 0.53 (HCT116), 0.51 (HEK293T) and 0.23 (K562); H1 0.01. Only the two Orion lines pass 0.4, and they share a lab and protocol, so that agreement may be a lab effect; H1 — Arc data, like A–F — agrees with none of the sources.
- **Decision:** skip the split for now and rely on edits 4–5. Revisit if the leaderboard shows A–C behaving unlike H1. The cheapest probe is a single submission with G = the average generic response of HCT116 / HEK293T / K562 added to every knockdown at full size (never DE-called).

The user confirmed preserving this skip decision for the current edit. The generic + specific build and G-on/off exam remain disabled; the benchmark tests calling and detection-floor settings.